# MiniMax-H3 (Base & 8-Step PDD Turbo) 3D 非均匀视频像素头 v2（Overcomplete + 统一帧轴精修）全云端一键训练台（Colab L4 24GB 专用）

本 Notebook 全程运行于 **Google Colab L4 (24GB VRAM)**，**100% 复用你网盘已有的 `minimax_h3_fl2va_pruned_w6a8` + `qwen3vl_32b_minimax_h3_nvfp4_awq` + 官方原版 `minimax_h3_video_vae_int8_convrot`（复用已有文件）**，一键训出高清锐利的 **`MiniMax-H3` 3D Video Pixel Head v2**：

- **v2 架构四大核心升级（修复 v1 轴错位模糊与 PDD 时间步偏移）**：
  1. **精确三级空间 `2× PixelShuffle`（`32px → 16px → 8px → 4px`）**：彻底修复 3D PixelShuffle 轴序，将单位置空间块缩小至精细 `4×4` 像素；
  2. **`(1, 4, 4, 4, 4)` 非均匀时序展开至统一像素帧轴 + 跨帧因果 3D 精修 + `5.33×` 过完备 FP32 头（`256 → 48`）**：在 `4px` 网格将 `t%5==0` (`1f`) 与 `t%5!=0` (`4f`) 汇合到连续 `F` 帧时间轴后再做因果 3D ConvNeXt 融合，共享同一个 `5.33×` 过完备输出头（仅 **`23.96M` 参数 / `~91.4 MiB` FP32**），消除 `1f/4f` 交界闪烁与 `256 → 768` 降秩模糊。
  3. **自适应 PDD 时间表校准（`video_shift = 12.0`）**：自动识别 `8-Step PDD Turbo` 8步 PDD 蒸馏头组（`n > 1`），将训练 $\sigma$ 精准锁定在 PDD 真实末步区间 `[0.444, 0.800]`（锚定 8步 `0.63158` 与 12步 `0.52174`），不再向 8步 PDD 模型喂入未蒸馏过的低 $\sigma$ 分布外噪声。
  4. **自动归档旧版 v1 断点**：自动备份旧 `latest_state.pt` 并复用已编码的 `/content/h3_ph_cache` 潜空间缓存。

In [ ]:
# @title 1. 挂载 Google Drive、校验 L4 显卡并自动部署 MIT 插件（致谢 sorryhyun/anima_lora）与对比工作流
import subprocess
from pathlib import Path
from google.colab import drive

subprocess.run(["nvidia-smi"], check=False)
drive.mount("/content/drive")

DRIVE_HUB = Path("/content/drive/MyDrive/ComfyUI_Hub")
PROJECT_DRIVE = Path("/content/drive/MyDrive/FastFrames_MiniMaxH3")
CKPT_DRIVE = PROJECT_DRIVE / "checkpoints"
EVAL_DRIVE = PROJECT_DRIVE / "eval_report"
PIXEL_HEADS_DRIVE = DRIVE_HUB / "models/pixel_heads"
VAE_DRIVE = DRIVE_HUB / "models/vae"
CUSTOM_NODE_DRIVE = DRIVE_HUB / "custom_nodes_h3/ComfyUI-MiniMaxH3-PixelHead"
WF_DRIVE_DIR = DRIVE_HUB / "user_h3/default/workflows"
LOCAL_SRC = Path("/content/ComfyUI-MiniMaxH3-PixelHead")

for d in (CKPT_DRIVE, EVAL_DRIVE, PIXEL_HEADS_DRIVE, VAE_DRIVE, CUSTOM_NODE_DRIVE, WF_DRIVE_DIR, LOCAL_SRC):
    d.mkdir(parents=True, exist_ok=True)

FILES = {'__init__.py': 'from .nodes import NODE_CLASS_MAPPINGS, NODE_DISPLAY_NAME_MAPPINGS\n\n__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS"]\n', 'LICENSE': 'MIT License\n\nCopyright (c) 2026\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'minimax_h3_pixel_head.py': '"""PyTorch implementation (adapted from sorryhyun/anima_lora & FastFrames) of 3D Non-Uniform Video Direct Pixel Head for MiniMax-H3.\n\nLicense: MIT\nArchitecture v2 (Overcomplete 3-Stage Spatial Shuffle + Unified Frame-Timeline Causal 3D Refinement)\nfollows FastFrames (arXiv:2611.23226), tailored specifically to MiniMax-H3 (`MiniMaxH3Model` + `MiniMaxH3VideoVAE`):\n  - Frozen MiniMax-H3 DiT (`hidden_size=5376`, `latents_dim=24`, `patch_size=(1, 2, 2)` -> `d_lat = 96`,\n    VAE `16x` spatial -> `32x32` pixels per DiT token; non-uniform `FRAME_PER_TOKEN = (1, 4, 4, 4, 4)` temporal!).\n  - Pre-hook on `diffusion_model.final_layer` computes the exact AdaLN-modulated video token sequence\n    `f_vid = mod(video_seg)` (`[B, T_lat * gh * gw, 5376]`), working identically on BOTH `n>1` PDD 8-Step Turbo\n    checkpoints (`8-Step PDD Turbo`) AND standard `n=1` Base/Ref2VA checkpoints!\n  - Progressive 3-stage Spatial PixelShuffle + Unified Frame-Level Causal 3D ConvNeXt:\n      1. `feat_norm(5376) -> Linear(5376 -> w32, bias=False)` + `Linear(96 -> w32, bias=True)` -> `entry_norm`.\n      2. `depth_s32` CausalConvNeXt3d blocks at `(T_lat, gh, gw)` (32px grid).\n      3. Spatial 2x PixelShuffle -> `depth_s16` CausalConvNeXt3d blocks at `(T_lat, 2*gh, 2*gw)` (16px grid).\n      4. Spatial 2x PixelShuffle -> `depth_s8` CausalConvNeXt3d blocks at `(T_lat, 4*gh, 4*gw)` (8px grid).\n      5. Spatial 2x PixelShuffle -> `(T_lat, 8*gh, 8*gw)` (4px grid).\n      6. Non-uniform `(1, 4, 4, 4, 4)` temporal unpack (`t1_proj` for `global_t % 5 == 0`, `t4_proj` for `!= 0`)\n         into the unified pixel-frame timeline `[B, w4, F_expected, 8*gh, 8*gw]`, followed by `depth_s4`\n         CausalConvNeXt3d blocks across adjacent frames (eliminating `1f <-> 4f` boundary seams!) and a single\n         shared `5.33x` overcomplete FP32 output head (`w4=256 -> 1 * 4 * 4 * 3 = 48`)!\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nFRAME_PER_TOKEN = (1, 4, 4, 4, 4)\nFINAL_SPATIAL_PATCH = 4  # After three 2x PixelShuffles: 32 / 8 = 4px\n\n\ndef h3_expected_frames(t_lat: int, t_offset: int = 0) -> int:\n    return sum(FRAME_PER_TOKEN[(t_offset + k) % 5] for k in range(t_lat))\n\n\nclass CausalDepthwiseConv3d(nn.Module):\n    """3D depthwise conv with left-only causal temporal padding and symmetric spatial padding."""\n\n    def __init__(self, dim: int, kt: int = 3, kh: int = 3, kw: int = 3):\n        super().__init__()\n        self.pad_t = kt - 1\n        self.pad_h = kh // 2\n        self.pad_w = kw // 2\n        self.conv = nn.Conv3d(dim, dim, kernel_size=(kt, kh, kw), padding=0, groups=dim)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if self.pad_t > 0:\n            first = x[:, :, :1].expand(-1, -1, self.pad_t, -1, -1)\n            x = torch.cat([first, x], dim=2)\n        if self.pad_h > 0 or self.pad_w > 0:\n            x = F.pad(x, (self.pad_w, self.pad_w, self.pad_h, self.pad_h, 0, 0), mode="replicate")\n        return self.conv(x)\n\n\nclass CausalConvNeXtBlock3d(nn.Module):\n    """3D Causal ConvNeXt block with channel LayerNorm, 4x GELU pointwise MLP, and LayerScale."""\n\n    def __init__(self, dim: int, mlp_ratio: int = 4, ls_init: float = 1.0):\n        super().__init__()\n        self.dw = CausalDepthwiseConv3d(dim, 3, 3, 3)\n        self.norm = nn.LayerNorm(dim, eps=1e-6)\n        hidden = dim * mlp_ratio\n        self.fc1 = nn.Linear(dim, hidden)\n        self.fc2 = nn.Linear(hidden, dim)\n        self.gamma = nn.Parameter(torch.full((dim,), float(ls_init)))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        h = self.dw(x).movedim(1, -1)\n        h = self.fc2(F.gelu(self.fc1(self.norm(h))))\n        h = (h * self.gamma).movedim(-1, 1).contiguous()\n        return x + h\n\n\ndef spatial_pixel_shuffle_3d(x: torch.Tensor, norm: nn.LayerNorm, proj: nn.Linear, scale: int = 2) -> torch.Tensor:\n    """Channel LayerNorm + Linear + exact per-frame `(1, scale, scale)` PixelShuffle on `[B, C, T, H, W]`."""\n    b, _, t, h, w = x.shape\n    u = proj(norm(x.movedim(1, -1)))  # [B, T, H, W, C_out * scale * scale]\n    c_out = u.shape[-1] // (scale * scale)\n    u = u.view(b, t, h, w, c_out, scale, scale)\n    return u.permute(0, 4, 1, 2, 5, 3, 6).contiguous().view(b, c_out, t, h * scale, w * scale)\n\n\ndef patchify_h3_video_latent(x0_hat_vid: torch.Tensor) -> tuple[torch.Tensor, int, int]:\n    """Convert `[B, 24, T_lat, H_lat, W_lat]` -> `([B, T_lat, gh, gw, 96], orig_px_h, orig_px_w)`."""\n    b, c, t_lat, h_lat, w_lat = x0_hat_vid.shape\n    orig_px_h, orig_px_w = h_lat * 16, w_lat * 16\n    pad_h = h_lat % 2\n    pad_w = w_lat % 2\n    if pad_h or pad_w:\n        x0_hat_vid = F.pad(x0_hat_vid, (0, pad_w, 0, pad_h, 0, 0), mode="replicate")\n    gh, gw = x0_hat_vid.shape[3] // 2, x0_hat_vid.shape[4] // 2\n    lat_tok = (\n        x0_hat_vid.view(b, c, t_lat, gh, 2, gw, 2)\n        .permute(0, 2, 3, 5, 1, 4, 6)\n        .contiguous()\n        .view(b, t_lat, gh, gw, c * 4)\n    )\n    return lat_tok, orig_px_h, orig_px_w\n\n\ndef unpatchify_h3_frames(\n    patches: torch.Tensor, ph: int = FINAL_SPATIAL_PATCH, pw: int = FINAL_SPATIAL_PATCH, out_channels: int = 3\n) -> torch.Tensor:\n    """Convert `[B, F, H8, W8, ph * pw * C]` -> `[B, C, F, H8 * ph, W8 * pw]`."""\n    b, f_cnt, h8, w8, _ = patches.shape\n    x = patches.view(b, f_cnt, h8, w8, ph, pw, out_channels)\n    x = x.permute(0, 6, 1, 2, 4, 3, 5).contiguous()\n    return x.view(b, out_channels, f_cnt, h8 * ph, w8 * pw)\n\n\nclass MiniMaxH3VideoPixelHead(nn.Module):\n    """Overcomplete 3D Non-Uniform Video Pixel Head for MiniMax-H3 (`(1,4,4,4,4)` frames, `32x32` spatial/token)."""\n\n    def __init__(\n        self,\n        d_feat: int = 5376,\n        d_lat: int = 96,\n        width: int = 768,\n        depth_s32: int = 2,\n        depth_s16: int = 2,\n        depth_s8: int = 2,\n        depth_s4: int = 1,\n        out_channels: int = 3,\n        ls_init: float = 1.0,\n    ):\n        super().__init__()\n        if width % 32 != 0:\n            raise ValueError(f"width must be divisible by 32, got {width}")\n        self.d_feat = int(d_feat)\n        self.d_lat = int(d_lat)\n        self.width = int(width)\n        self.depth_s32 = int(depth_s32)\n        self.depth_s16 = int(depth_s16)\n        self.depth_s8 = int(depth_s8)\n        self.depth_s4 = int(depth_s4)\n        self.out_channels = int(out_channels)\n        self.ls_init = float(ls_init)\n\n        w32 = width\n        w16 = max(128, ((width * 2) // 3 // 32) * 32)   # 512 when width=768\n        w8 = max(128, (width // 2 // 32) * 32)          # 384 when width=768\n        w4 = max(128, (width // 3 // 32) * 32)          # 256 when width=768 (>> 48 out_dim!)\n        self.w4 = w4\n\n        self.feat_norm = nn.LayerNorm(self.d_feat, eps=1e-6)\n        self.feat_proj = nn.Linear(self.d_feat, w32, bias=False)\n        self.lat_proj = nn.Linear(self.d_lat, w32, bias=True)\n        self.entry_norm = nn.LayerNorm(w32, eps=1e-6)\n\n        self.s32_blocks = nn.ModuleList([CausalConvNeXtBlock3d(w32, ls_init=ls_init) for _ in range(depth_s32)])\n        self.up1_norm = nn.LayerNorm(w32, eps=1e-6)\n        self.up1_proj = nn.Linear(w32, 4 * w16)\n\n        self.s16_blocks = nn.ModuleList([CausalConvNeXtBlock3d(w16, ls_init=ls_init) for _ in range(depth_s16)])\n        self.up2_norm = nn.LayerNorm(w16, eps=1e-6)\n        self.up2_proj = nn.Linear(w16, 4 * w8)\n\n        self.s8_blocks = nn.ModuleList([CausalConvNeXtBlock3d(w8, ls_init=ls_init) for _ in range(depth_s8)])\n        self.up3_norm = nn.LayerNorm(w8, eps=1e-6)\n        self.up3_proj = nn.Linear(w8, 4 * w4)\n\n        # Non-uniform (1, 4, 4, 4, 4) temporal expansion onto unified frame grid at 4px spatial resolution\n        self.t_norm = nn.LayerNorm(w4, eps=1e-6)\n        self.t1_proj = nn.Linear(w4, 1 * w4)\n        self.t4_proj = nn.Linear(w4, 4 * w4)\n\n        # Unified frame-level causal 3D refinement across all F_expected frames (zero 1f<->4f seam!)\n        self.s4_blocks = nn.ModuleList([CausalConvNeXtBlock3d(w4, ls_init=ls_init) for _ in range(depth_s4)])\n        self.norm_out = nn.LayerNorm(w4, eps=1e-6)\n\n        out_dim = FINAL_SPATIAL_PATCH * FINAL_SPATIAL_PATCH * self.out_channels  # 4*4*3 = 48\n        self.out_proj = nn.Linear(w4, out_dim)\n        nn.init.normal_(self.out_proj.weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.out_proj.bias)\n\n        self.meta: dict[str, str] = {}\n\n    def causal_receptive_t_lat(self) -> int:\n        """Exact causal temporal lookback in latent frames (`T_lat`) for zero-seam chunked decoding."""\n        return 2 * self.depth_s32 + 2 * self.depth_s16 + 2 * self.depth_s8 + 2 * self.depth_s4\n\n    def _expand_nonuniform_time(self, h_s4: torch.Tensor, t_offset: int) -> torch.Tensor:\n        """Expand `[B, w4, T_lat, H8, W8]` -> `[B, w4, F_expected, H8, W8]` according to `(1, 4, 4, 4, 4)`."""\n        b, c4, t_lat, h8, w8 = h_s4.shape\n        u = self.t_norm(h_s4.movedim(1, -1))  # [B, T_lat, H8, W8, w4]\n\n        idx_1f = [i for i in range(t_lat) if (t_offset + i) % 5 == 0]\n        idx_4f = [i for i in range(t_lat) if (t_offset + i) % 5 != 0]\n\n        slots: list[Optional[torch.Tensor]] = [None] * t_lat\n        if idx_1f:\n            # [B, n1, H8, W8, w4] -> [B, w4, n1, H8, W8]\n            y1 = self.t1_proj(u[:, idx_1f]).permute(0, 4, 1, 2, 3).contiguous()\n            for k, ti in enumerate(idx_1f):\n                slots[ti] = y1[:, :, k : k + 1]\n        if idx_4f:\n            # [B, n4, H8, W8, w4 * 4] -> [B, n4, H8, W8, w4, 4] -> [B, w4, n4, 4, H8, W8]\n            y4 = self.t4_proj(u[:, idx_4f]).view(b, len(idx_4f), h8, w8, c4, 4).permute(0, 4, 1, 5, 2, 3).contiguous()\n            for k, ti in enumerate(idx_4f):\n                slots[ti] = y4[:, :, k]  # [B, w4, 4, H8, W8]\n\n        return torch.cat([s for s in slots if s is not None], dim=2)\n\n    def forward(\n        self,\n        feat: torch.Tensor,\n        x0_hat_vid: torch.Tensor,\n        t_offset: int = 0,\n    ) -> torch.Tensor:\n        """Decode `(feat, x0_hat_vid)` into `[B, 3, F_expected, H_px, W_px]` in `[-1, 1]`."""\n        if x0_hat_vid.ndim == 5 and x0_hat_vid.shape[1] == self.d_lat:\n            b, _, t_lat, gh, gw = x0_hat_vid.shape\n            lat_tok = x0_hat_vid.movedim(1, -1)\n            orig_px_h, orig_px_w = gh * 32, gw * 32\n        else:\n            lat_tok, orig_px_h, orig_px_w = patchify_h3_video_latent(x0_hat_vid)\n            b, t_lat, gh, gw, _ = lat_tok.shape\n\n        if feat.ndim == 3:\n            feat = feat.view(b, t_lat, gh, gw, self.d_feat)\n\n        h = self.entry_norm(self.feat_proj(self.feat_norm(feat)) + self.lat_proj(lat_tok))\n        h = h.movedim(-1, 1).contiguous()  # [B, w32, T_lat, gh, gw]\n\n        for blk in self.s32_blocks:\n            h = blk(h)\n        h = spatial_pixel_shuffle_3d(h, self.up1_norm, self.up1_proj, scale=2)\n        for blk in self.s16_blocks:\n            h = blk(h)\n        h = spatial_pixel_shuffle_3d(h, self.up2_norm, self.up2_proj, scale=2)\n        for blk in self.s8_blocks:\n            h = blk(h)\n        h = spatial_pixel_shuffle_3d(h, self.up3_norm, self.up3_proj, scale=2)  # [B, w4, T_lat, 8*gh, 8*gw]\n\n        h_frames = self._expand_nonuniform_time(h, t_offset=t_offset)  # [B, w4, F_expected, 8*gh, 8*gw]\n        for blk in self.s4_blocks:\n            h_frames = blk(h_frames)\n\n        h_tok = self.norm_out(h_frames.movedim(1, -1))  # [B, F_expected, 8*gh, 8*gw, w4]\n        with torch.autocast(device_type=h_tok.device.type, enabled=False):\n            patches = self.out_proj(h_tok.float())\n        vid = unpatchify_h3_frames(patches, FINAL_SPATIAL_PATCH, FINAL_SPATIAL_PATCH, self.out_channels)\n        return vid[:, :, :, :orig_px_h, :orig_px_w]\n\n    @torch.no_grad()\n    def decode_chunked(\n        self,\n        feat: torch.Tensor,\n        x0_hat_vid: torch.Tensor,\n        chunk_t: int = 10,\n        overlap_t: Optional[int] = None,\n    ) -> torch.Tensor:\n        """Low-VRAM bit-exact causal chunked decode for 124~362+ frame MiniMax-H3 videos."""\n        lat_tok, orig_px_h, orig_px_w = patchify_h3_video_latent(x0_hat_vid)\n        b, t_lat, gh, gw, _ = lat_tok.shape\n        if feat.ndim == 3:\n            feat = feat.view(b, t_lat, gh, gw, self.d_feat)\n        lat_5d = lat_tok.movedim(-1, 1).contiguous()  # [B, 96, T_lat, gh, gw]\n\n        if t_lat <= chunk_t:\n            out = self.forward(feat, lat_5d, t_offset=0)\n            return out[:, :, :, :orig_px_h, :orig_px_w]\n\n        ov = self.causal_receptive_t_lat() if overlap_t is None else int(overlap_t)\n        pieces = []\n        pos = 0\n        while pos < t_lat:\n            end = min(t_lat, pos + chunk_t)\n            ctx_start = max(0, pos - ov)\n            f_sl = feat[:, ctx_start:end]\n            x_sl = lat_5d[:, :, ctx_start:end]\n            dec_sl = self.forward(f_sl, x_sl, t_offset=ctx_start)[:, :, :, :orig_px_h, :orig_px_w]\n            if pos == 0:\n                pieces.append(dec_sl.cpu())\n            else:\n                drop_frames = h3_expected_frames(pos - ctx_start, t_offset=ctx_start)\n                pieces.append(dec_sl[:, :, drop_frames:].cpu())\n            pos = end\n        return torch.cat(pieces, dim=2)\n\n\ndef save_minimax_h3_pixel_head(\n    head: MiniMaxH3VideoPixelHead,\n    path: str | Path,\n    extra_meta: Optional[dict[str, str]] = None,\n) -> None:\n    from safetensors.torch import save_file\n\n    meta = {\n        "arch": "minimax_h3_video_pixel_head_v2",\n        "d_feat": str(head.d_feat),\n        "d_lat": str(head.d_lat),\n        "width": str(head.width),\n        "depth_s32": str(head.depth_s32),\n        "depth_s16": str(head.depth_s16),\n        "depth_s8": str(head.depth_s8),\n        "depth_s4": str(head.depth_s4),\n        "out_channels": str(head.out_channels),\n        "ls_init": str(head.ls_init),\n        "frame_per_token": "1,4,4,4,4",\n        "latent_input": "x0_hat_vid_1x2x2",\n    }\n    if extra_meta:\n        for k, v in extra_meta.items():\n            meta[str(k)] = str(v)\n\n    sd = {k: v.detach().cpu().contiguous() for k, v in head.state_dict().items()}\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    save_file(sd, str(path), metadata=meta)\n\n\ndef load_minimax_h3_pixel_head(\n    path: str | Path,\n    device: str | torch.device = "cpu",\n    dtype: torch.dtype = torch.float32,\n) -> MiniMaxH3VideoPixelHead:\n    from safetensors import safe_open\n    from safetensors.torch import load_file\n\n    path = Path(path)\n    with safe_open(str(path), framework="pt", device="cpu") as fh:\n        meta = dict(fh.metadata() or {})\n\n    model = MiniMaxH3VideoPixelHead(\n        d_feat=int(meta.get("d_feat", "5376")),\n        d_lat=int(meta.get("d_lat", "96")),\n        width=int(meta.get("width", "768")),\n        depth_s32=int(meta.get("depth_s32", "2")),\n        depth_s16=int(meta.get("depth_s16", "2")),\n        depth_s8=int(meta.get("depth_s8", "2")),\n        depth_s4=int(meta.get("depth_s4", "1")),\n        out_channels=int(meta.get("out_channels", "3")),\n        ls_init=float(meta.get("ls_init", "1.0")),\n    )\n    sd = load_file(str(path), device="cpu")\n    model.load_state_dict(sd, strict=True)\n    model = model.to(device=device, dtype=dtype)\n    model.out_proj.float()\n    model.meta = meta\n    model.eval()\n    return model\n', 'nodes.py': '"""ComfyUI Nodes for MiniMax-H3 3D Video Direct Pixel Head (Zero-Video-VAE Decoding).\n\nLicense: MIT\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport time\nimport logging\n\nimport folder_paths\nimport comfy.model_management\nimport torch\n\nfrom .minimax_h3_pixel_head import load_minimax_h3_pixel_head\nfrom .tap import attach_minimax_h3_tap\n\nlogger = logging.getLogger(__name__)\n\nPIXEL_HEAD_DIR = os.path.join(folder_paths.models_dir, "pixel_heads")\nos.makedirs(PIXEL_HEAD_DIR, exist_ok=True)\nif "pixel_heads" not in folder_paths.folder_names_and_paths:\n    folder_paths.add_model_folder_path("pixel_heads", PIXEL_HEAD_DIR)\n\n\ndef _list_candidate_files() -> list[str]:\n    seen = set()\n    items = []\n    for folder_key in ("pixel_heads", "vae"):\n        if folder_key in folder_paths.folder_names_and_paths:\n            for fn in folder_paths.get_filename_list(folder_key):\n                if fn.endswith(".safetensors") and fn not in seen:\n                    seen.add(fn)\n                    items.append(fn)\n    return sorted(items) if items else ["minimax_h3_video_pixel_head.safetensors"]\n\n\ndef _resolve_head_path(name: str) -> str:\n    for folder_key in ("pixel_heads", "vae"):\n        if folder_key in folder_paths.folder_names_and_paths:\n            p = folder_paths.get_full_path(folder_key, name)\n            if p and os.path.exists(p):\n                return p\n    raise FileNotFoundError(\n        f"未找到 MiniMax-H3 Pixel Head 权重 `{name}`，请将其放入 `ComfyUI/models/pixel_heads/` 或 `ComfyUI/models/vae/`。"\n    )\n\n\nclass MiniMaxH3PixelHeadLoader:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "head_name": (_list_candidate_files(),),\n                "precision": (["bf16_trunk_fp32_out", "fp32"], {"default": "bf16_trunk_fp32_out"}),\n            }\n        }\n\n    RETURN_TYPES = ("MINIMAX_H3_PIXEL_HEAD",)\n    RETURN_NAMES = ("pixel_head",)\n    FUNCTION = "load_head"\n    CATEGORY = "advanced/minimax_h3"\n    TITLE = "Load MiniMax-H3 Video Pixel Head"\n\n    def load_head(self, head_name: str, precision: str):\n        path = _resolve_head_path(head_name)\n        dtype = torch.bfloat16 if precision == "bf16_trunk_fp32_out" else torch.float32\n        head = load_minimax_h3_pixel_head(path, device="cpu", dtype=dtype)\n        return (head,)\n\n\nclass MiniMaxH3PixelHeadTap:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "model": ("MODEL",),\n            }\n        }\n\n    RETURN_TYPES = ("MODEL", "MINIMAX_H3_TAP")\n    RETURN_NAMES = ("model", "pixel_tap")\n    FUNCTION = "install_tap"\n    CATEGORY = "advanced/minimax_h3"\n    TITLE = "MiniMax-H3 Video Pixel Head Tap"\n\n    def install_tap(self, model):\n        patched_model, tap_state = attach_minimax_h3_tap(model)\n        return (patched_model, tap_state)\n\n\nclass MiniMaxH3VideoPixelDecode:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "samples": ("LATENT",),\n                "pixel_head": ("MINIMAX_H3_PIXEL_HEAD",),\n                "pixel_tap": ("MINIMAX_H3_TAP",),\n                "temporal_chunk_latents": ("INT", {"default": 10, "min": 5, "max": 60, "step": 5}),\n            }\n        }\n\n    RETURN_TYPES = ("IMAGE",)\n    RETURN_NAMES = ("images",)\n    FUNCTION = "decode"\n    CATEGORY = "advanced/minimax_h3"\n    TITLE = "MiniMax-H3 Video Pixel Decode (Zero Video VAE)"\n\n    @torch.no_grad()\n    def decode(self, samples, pixel_head, pixel_tap, temporal_chunk_latents: int = 10):\n        _ = samples["samples"]\n        f_cpu, x0_vid_cpu, sigma = pixel_tap.resolve_final_features()\n\n        device = comfy.model_management.get_torch_device()\n        head_dev = pixel_head.to(device)\n        trunk_dtype = next(head_dev.parameters()).dtype\n        f_dev = f_cpu.to(device=device, dtype=trunk_dtype)\n        x_dev = x0_vid_cpu.to(device=device, dtype=trunk_dtype)\n\n        if torch.cuda.is_available() and device.type == "cuda":\n            torch.cuda.synchronize(device)\n        t0 = time.perf_counter()\n\n        out_bcthw = head_dev.decode_chunked(f_dev, x_dev, chunk_t=int(temporal_chunk_latents))\n\n        if torch.cuda.is_available() and device.type == "cuda":\n            torch.cuda.synchronize(device)\n        dt_ms = (time.perf_counter() - t0) * 1000.0\n\n        b, c, f_cnt, h, w = out_bcthw.shape\n        frames = (\n            ((out_bcthw.float() + 1.0) * 0.5)\n            .clamp_(0.0, 1.0)\n            .permute(0, 2, 3, 4, 1)\n            .reshape(b * f_cnt, h, w, c)\n            .contiguous()\n            .cpu()\n        )\n        pixel_head.to("cpu")\n\n        logger.info(\n            f"[MiniMaxH3VideoPixelDecode] Decoded {f_cnt} frames @ {w}x{h} (last_sigma={sigma:.4f}) in {dt_ms:.1f} ms (Zero Video VAE)"\n        )\n        return (frames,)\n\n\nNODE_CLASS_MAPPINGS = {\n    "MiniMaxH3PixelHeadLoader": MiniMaxH3PixelHeadLoader,\n    "MiniMaxH3PixelHeadTap": MiniMaxH3PixelHeadTap,\n    "MiniMaxH3VideoPixelDecode": MiniMaxH3VideoPixelDecode,\n}\n\nNODE_DISPLAY_NAME_MAPPINGS = {\n    "MiniMaxH3PixelHeadLoader": "Load MiniMax-H3 Video Pixel Head",\n    "MiniMaxH3PixelHeadTap": "MiniMax-H3 Video Pixel Head Tap",\n    "MiniMaxH3VideoPixelDecode": "MiniMax-H3 Video Pixel Decode (Zero Video VAE)",\n}\n', 'README.md': '# ComfyUI-MiniMaxH3-PixelHead\n\nZero-Video-VAE 3D Non-Uniform Direct Pixel Readout Head (`v2` Overcomplete 3-Stage Spatial Shuffle + Unified Frame-Timeline 3D Refinement) for **MiniMax-H3 (Base & 8-Step PDD Turbo)** (`MiniMaxH3Model`) in ComfyUI.\n\nTaps `diffusion_model.final_layer` (`5376`-dim `mod(video_seg)` tokens — compatible with both `8-Step PDD Turbo` 8-Step PDD Turbo and standard Base/Ref2VA checkpoints) + denoised `24`-ch `(1,2,2)` video latent (`d_lat=96`), using three `2x` spatial PixelShuffle stages (`32px -> 16px -> 8px -> 4px`), non-uniform `(1, 4, 4, 4, 4)` temporal expansion onto the unified frame timeline, frame-level causal 3D ConvNeXt refinement, and a `5.33x` overcomplete FP32 head (`256 -> 48`) to replace the multi-GB 36-layer ViT3D `MiniMaxH3VideoVAE`.\n\n## License\nMIT License (retaining Copyright (c) 2026 Seunghyun Ji / sorryhyun).\n', 'tap.py': '"""Non-invasive ComfyUI ModelPatcher tap for MiniMax-H3 (`comfy.ldm.minimax.model.MiniMaxH3Model`).\n\nLicense: MIT\nWhy we hook `diffusion_model.final_layer` instead of `final_layer.video_out`:\n  - In `comfy/ldm/minimax/model.py` lines 322-336, PDD 8-Step Turbo checkpoints (`minimax_h3_8stepTURBO...`, where `n > 1`)\n    call `_pdd_head(self.video_out, mod(video_seg), ...)` using `F.linear` directly on `video_out.weight`, bypassing\n    `video_out.forward`!\n  - By registering a `register_forward_pre_hook` on `diffusion_model.final_layer` itself (`inputs = (h, t_emb, video_seg, ...)`),\n    we compute the exact 3-line `mod(video_seg)`:\n      `shift, scale = fl.adaln_proj(t_emb)`\n      `f_vid = fl.norm(h[va:vb]) * (1.0 + scale[row]) + shift[row]`\n    This works 100% identically across BOTH `8-Step PDD Turbo` 8-Step PDD Turbo (`n > 1`) AND standard `n = 1` Base / Ref2VA\n    checkpoints, while automatically stripping text, keyframe, reference image, and audio tokens via `video_seg = (va, vb, row)`!\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nimport math\nfrom typing import Optional\n\nimport comfy.patcher_extension\nimport torch\n\n\nTAP_WRAPPER_KEY = "minimax_h3_pixel_head_tap"\n\n\ndef _extract_h3_video_latent(tensor: torch.Tensor, vid_shape: Optional[tuple[int, ...]]) -> torch.Tensor:\n    if getattr(tensor, "is_nested", False):\n        return tensor.unbind()[0]\n    if tensor.ndim == 5:\n        return tensor\n    if vid_shape is not None:\n        vid_elems = int(math.prod(vid_shape[1:]))\n        b = tensor.shape[0]\n        return tensor.reshape(b, -1)[:, :vid_elems].reshape(b, *vid_shape[1:])\n    raise ValueError(f"Cannot unpack MiniMax-H3 video latent from shape {tuple(tensor.shape)} without latent_shapes")\n\n\n@dataclass\nclass StepRecord:\n    sigma: float\n    vid_shape: Optional[tuple[int, ...]] = None\n    forwards: list[dict] = field(default_factory=list)\n    cond_scale: Optional[float] = None\n    denoised_vid: Optional[torch.Tensor] = None\n\n\nclass MiniMaxH3TapState:\n    """Stores only the latest actual MiniMax-H3 DiT forward step\'s video features on CPU."""\n\n    def __init__(self):\n        self.current_step: Optional[StepRecord] = None\n        self.last_valid_step: Optional[StepRecord] = None\n\n    def open_step(self, sigma: float) -> None:\n        self.current_step = StepRecord(sigma=sigma)\n\n    def close_step(self) -> None:\n        if self.current_step is not None and len(self.current_step.forwards) > 0:\n            self.last_valid_step = self.current_step\n        self.current_step = None\n\n    def resolve_final_features(self) -> tuple[torch.Tensor, torch.Tensor, float]:\n        """Return `(f_vid [B, T_lat*gh*gw, 5376], x0_hat_vid [B, 24, T_lat, H_lat, W_lat], sigma)`."""\n        step = self.last_valid_step\n        if step is None or not step.forwards:\n            raise RuntimeError(\n                "MiniMaxH3VideoPixelDecode: 未捕获到任何前向特征！请确认已将 `MiniMax-H3 Pixel Head Tap` "\n                "串联在进入 BasicGuider / KSampler 的 MODEL 连线上（位于 MiniMaxH3SigmaShift 之后）。"\n            )\n\n        by_tag: dict[int, list[tuple[torch.Tensor, torch.Tensor]]] = {0: [], 1: []}\n        for rec in step.forwards:\n            cou = rec["cond_or_uncond"]\n            n_chunks = len(cou)\n            f_chunks = rec["f"].chunk(n_chunks, dim=0)\n            d_chunks = rec["denoised_vid"].chunk(n_chunks, dim=0)\n            for tag, fc, dc in zip(cou, f_chunks, d_chunks):\n                by_tag.setdefault(int(tag), []).append((fc, dc))\n\n        if not by_tag.get(0):\n            raise RuntimeError("MiniMaxH3VideoPixelDecode: 最后一步未找到正向条件 (cond=0) 特征记录。")\n\n        f_cond = torch.cat([p[0] for p in by_tag[0]], dim=0).float()\n        d_cond = torch.cat([p[1] for p in by_tag[0]], dim=0).float()\n\n        if by_tag.get(1) and step.cond_scale is not None and abs(step.cond_scale - 1.0) > 1e-5:\n            f_uncond = torch.cat([p[0] for p in by_tag[1]], dim=0).float()\n            d_uncond = torch.cat([p[1] for p in by_tag[1]], dim=0).float()\n            scale = float(step.cond_scale)\n            f_final = f_uncond + scale * (f_cond - f_uncond)\n            x0_hat_fallback = d_uncond + scale * (d_cond - d_uncond)\n        else:\n            f_final = f_cond\n            x0_hat_fallback = d_cond\n\n        x0_hat_vid = step.denoised_vid.float() if step.denoised_vid is not None else x0_hat_fallback\n        return f_final, x0_hat_vid, float(step.sigma)\n\n\ndef attach_minimax_h3_tap(model_patcher) -> tuple[object, MiniMaxH3TapState]:\n    """Clone `model_patcher` and install lightweight step + `final_layer` pre-hooks for MiniMax-H3."""\n    cloned = model_patcher.clone()\n    state = MiniMaxH3TapState()\n\n    cloned.remove_wrappers_with_key(comfy.patcher_extension.WrappersMP.CALC_COND_BATCH, TAP_WRAPPER_KEY)\n    cloned.remove_wrappers_with_key(comfy.patcher_extension.WrappersMP.APPLY_MODEL, TAP_WRAPPER_KEY)\n\n    def calc_cond_batch_wrapper(executor, model, conds, x_in, timestep, model_options):\n        sig = float(timestep.flatten()[0].detach().cpu().item())\n        state.open_step(sig)\n        try:\n            return executor(model, conds, x_in, timestep, model_options)\n        finally:\n            state.close_step()\n\n    def apply_model_wrapper(executor, x, t, c_concat=None, c_crossattn=None, control=None, transformer_options={}, **kwargs):\n        step = state.current_step\n        cou = (transformer_options or {}).get("cond_or_uncond")\n        if step is None or not cou:\n            return executor(x, t, c_concat, c_crossattn, control, transformer_options, **kwargs)\n\n        base_model = executor.class_obj\n        diff_model = getattr(base_model, "diffusion_model", None)\n        final_layer = getattr(diff_model, "final_layer", None)\n        if final_layer is None or not hasattr(final_layer, "adaln_proj"):\n            raise RuntimeError(\n                f"MiniMaxH3PixelHeadTap: 当前模型 ({type(diff_model).__name__}) 不包含 MiniMax-H3 `final_layer`，请确认底模为 MiniMax-H3 (Base & 8-Step PDD Turbo)。"\n            )\n\n        latent_shapes = kwargs.get("latent_shapes", getattr(base_model, "latent_shapes", None))\n        if latent_shapes is not None and len(latent_shapes) > 0:\n            vid_shape = tuple(int(s) for s in latent_shapes[0])\n        elif x.ndim == 5:\n            vid_shape = tuple(int(s) for s in x.shape)\n        else:\n            vid_shape = None\n        step.vid_shape = vid_shape\n\n        b_call = int(x.shape[0])\n        rec = {\n            "denoised_vid": None,\n            "f": None,\n            "cond_or_uncond": list(cou),\n        }\n\n        def _pre_final_hook(fl, inputs):\n            if rec["f"] is None and len(inputs) >= 3:\n                with torch.no_grad():\n                    h, t_emb, video_seg = inputs[0], inputs[1], inputs[2]\n                    va, vb, row = video_seg\n                    shift, scale = fl.adaln_proj(t_emb)\n                    sc = scale[row].to(scale.dtype)\n                    sh = shift[row].to(shift.dtype)\n                    f_vid = fl.norm(h[va:vb]) * (1.0 + sc) + sh\n                    rec["f"] = f_vid.reshape(b_call, -1, f_vid.shape[-1]).detach().to(device="cpu", dtype=torch.bfloat16)\n\n        handle = final_layer.register_forward_pre_hook(_pre_final_hook)\n        try:\n            out = executor(x, t, c_concat, c_crossattn, control, transformer_options, **kwargs)\n        finally:\n            handle.remove()\n\n        if rec["f"] is not None:\n            d_vid = _extract_h3_video_latent(out, vid_shape)\n            rec["denoised_vid"] = d_vid.detach().to(device="cpu", dtype=torch.float32)\n            step.forwards.append(rec)\n        return out\n\n    def post_cfg_hook(args):\n        target_step = state.current_step if state.current_step is not None else state.last_valid_step\n        if target_step is not None:\n            if "cond_scale" in args and isinstance(args["cond_scale"], (int, float)):\n                target_step.cond_scale = float(args["cond_scale"])\n            if "denoised" in args and args["denoised"] is not None:\n                try:\n                    d_vid = _extract_h3_video_latent(args["denoised"], target_step.vid_shape)\n                    target_step.denoised_vid = d_vid.detach().to(device="cpu", dtype=torch.float32)\n                except Exception:\n                    pass\n        return args["denoised"]\n\n    cloned.add_wrapper_with_key(\n        comfy.patcher_extension.WrappersMP.CALC_COND_BATCH,\n        TAP_WRAPPER_KEY,\n        calc_cond_batch_wrapper,\n    )\n    cloned.add_wrapper_with_key(\n        comfy.patcher_extension.WrappersMP.APPLY_MODEL,\n        TAP_WRAPPER_KEY,\n        apply_model_wrapper,\n    )\n    cloned.set_model_sampler_post_cfg_function(post_cfg_hook)\n    cloned.model_options[TAP_WRAPPER_KEY] = state\n    return cloned, state\n'}
for fname, content in FILES.items():
    (LOCAL_SRC / fname).write_text(content, encoding="utf-8")
    (CUSTOM_NODE_DRIVE / fname).write_text(content, encoding="utf-8")

(WF_DRIVE_DIR / "07_MiniMaxH3_VideoPixelHead_免VAE极速视频直出与对比.json").write_text('{\n  "last_node_id": 14,\n  "last_link_id": 15,\n  "nodes": [\n    {\n      "id": 1,\n      "type": "MarkdownNote",\n      "pos": [\n        40,\n        -210\n      ],\n      "size": [\n        1460,\n        175\n      ],\n      "order": 0,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [],\n      "title": "⚡ MiniMax-H3 (Base & 8-Step PDD Turbo) Video PixelHead：彻底省去 2.8~5.2GB ViT3D Video VAE！",\n      "widgets_values": [\n        "### MiniMax-H3 Direct 3D Video Pixel Head（Colab L4 24GB 专用）\\n- **省去 36 层 ViT3D 重型 Video VAE**：将 `MiniMaxH3PixelHeadTap` 挂在 `MiniMaxH3SigmaShift` 之后，采样结束由 `MiniMaxH3VideoPixelDecode` 按 `(1,4,4,4,4)` 精确还原 `17k+5` 帧视频（纯文生 T2VA 甚至无需加载 Video VAE）！"\n      ]\n    },\n    {\n      "id": 2,\n      "type": "UNETLoader",\n      "pos": [\n        40,\n        20\n      ],\n      "size": [\n        440,\n        88\n      ],\n      "order": 1,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            1\n          ]\n        }\n      ],\n      "widgets_values": [\n        "minimax_h3_fl2va_pruned_w6a8.safetensors",\n        "default"\n      ]\n    },\n    {\n      "id": 3,\n      "type": "CLIPLoaderGGUF",\n      "pos": [\n        40,\n        145\n      ],\n      "size": [\n        440,\n        92\n      ],\n      "order": 2,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "CLIP",\n          "type": "CLIP",\n          "links": [\n            4\n          ]\n        }\n      ],\n      "widgets_values": [\n        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",\n        "minimax"\n      ]\n    },\n    {\n      "id": 4,\n      "type": "MiniMaxH3SigmaShift",\n      "pos": [\n        40,\n        270\n      ],\n      "size": [\n        440,\n        86\n      ],\n      "order": 3,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 1\n        }\n      ],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            2\n          ]\n        }\n      ],\n      "widgets_values": [\n        12.0,\n        3.0\n      ]\n    },\n    {\n      "id": 5,\n      "type": "MiniMaxH3PixelHeadLoader",\n      "pos": [\n        40,\n        390\n      ],\n      "size": [\n        440,\n        88\n      ],\n      "order": 4,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "pixel_head",\n          "type": "MINIMAX_H3_PIXEL_HEAD",\n          "links": [\n            13\n          ]\n        }\n      ],\n      "widgets_values": [\n        "minimax_h3_video_pixel_head.safetensors",\n        "bf16_trunk_fp32_out"\n      ]\n    },\n    {\n      "id": 6,\n      "type": "MiniMaxH3PixelHeadTap",\n      "pos": [\n        40,\n        515\n      ],\n      "size": [\n        440,\n        68\n      ],\n      "order": 5,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 2\n        }\n      ],\n      "outputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "links": [\n            3,\n            9\n          ]\n        },\n        {\n          "name": "pixel_tap",\n          "type": "MINIMAX_H3_TAP",\n          "links": [\n            14\n          ]\n        }\n      ]\n    },\n    {\n      "id": 7,\n      "type": "CLIPTextEncode",\n      "pos": [\n        520,\n        20\n      ],\n      "size": [\n        440,\n        200\n      ],\n      "order": 6,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 4\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            5\n          ]\n        }\n      ],\n      "widgets_values": [\n        "Cinematic 24fps anime film shot of a young swordsman standing on a windy cliff overlooking sunlit clouds, hair and cloak flowing naturally in the breeze, smooth camera push-in. Audio: gentle mountain wind, fluttering cloth."\n      ]\n    },\n    {\n      "id": 8,\n      "type": "EmptyMiniMaxH3LatentAV",\n      "pos": [\n        520,\n        260\n      ],\n      "size": [\n        440,\n        110\n      ],\n      "order": 7,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            11\n          ]\n        }\n      ],\n      "widgets_values": [\n        896,\n        512,\n        124\n      ]\n    },\n    {\n      "id": 9,\n      "type": "BasicGuider",\n      "pos": [\n        1000,\n        20\n      ],\n      "size": [\n        340,\n        68\n      ],\n      "order": 8,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 3\n        },\n        {\n          "name": "conditioning",\n          "type": "CONDITIONING",\n          "link": 5\n        }\n      ],\n      "outputs": [\n        {\n          "name": "GUIDER",\n          "type": "GUIDER",\n          "links": [\n            7\n          ]\n        }\n      ]\n    },\n    {\n      "id": 10,\n      "type": "RandomNoise",\n      "pos": [\n        1000,\n        115\n      ],\n      "size": [\n        165,\n        80\n      ],\n      "order": 9,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "NOISE",\n          "type": "NOISE",\n          "links": [\n            6\n          ]\n        }\n      ],\n      "widgets_values": [\n        20261122,\n        "fixed"\n      ]\n    },\n    {\n      "id": 11,\n      "type": "KSamplerSelect",\n      "pos": [\n        1175,\n        115\n      ],\n      "size": [\n        165,\n        80\n      ],\n      "order": 10,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "SAMPLER",\n          "type": "SAMPLER",\n          "links": [\n            8\n          ]\n        }\n      ],\n      "widgets_values": [\n        "euler"\n      ]\n    },\n    {\n      "id": 12,\n      "type": "BasicScheduler",\n      "pos": [\n        1000,\n        220\n      ],\n      "size": [\n        340,\n        106\n      ],\n      "order": 11,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 9\n        }\n      ],\n      "outputs": [\n        {\n          "name": "SIGMAS",\n          "type": "SIGMAS",\n          "links": [\n            10\n          ]\n        }\n      ],\n      "widgets_values": [\n        "simple",\n        8,\n        1.0\n      ]\n    },\n    {\n      "id": 13,\n      "type": "SamplerCustomAdvanced",\n      "pos": [\n        1000,\n        355\n      ],\n      "size": [\n        340,\n        120\n      ],\n      "order": 12,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "noise",\n          "type": "NOISE",\n          "link": 6\n        },\n        {\n          "name": "guider",\n          "type": "GUIDER",\n          "link": 7\n        },\n        {\n          "name": "sampler",\n          "type": "SAMPLER",\n          "link": 8\n        },\n        {\n          "name": "sigmas",\n          "type": "SIGMAS",\n          "link": 10\n        },\n        {\n          "name": "latent_image",\n          "type": "LATENT",\n          "link": 11\n        }\n      ],\n      "outputs": [\n        {\n          "name": "output",\n          "type": "LATENT",\n          "links": [\n            12\n          ]\n        },\n        {\n          "name": "denoised_output",\n          "type": "LATENT",\n          "links": []\n        }\n      ]\n    },\n    {\n      "id": 14,\n      "type": "MiniMaxH3VideoPixelDecode",\n      "pos": [\n        1000,\n        505\n      ],\n      "size": [\n        340,\n        120\n      ],\n      "order": 13,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 12\n        },\n        {\n          "name": "pixel_head",\n          "type": "MINIMAX_H3_PIXEL_HEAD",\n          "link": 13\n        },\n        {\n          "name": "pixel_tap",\n          "type": "MINIMAX_H3_TAP",\n          "link": 14\n        }\n      ],\n      "outputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "links": []\n        }\n      ],\n      "widgets_values": [\n        10\n      ]\n    }\n  ],\n  "links": [\n    [\n      1,\n      2,\n      0,\n      4,\n      0,\n      "MODEL"\n    ],\n    [\n      2,\n      4,\n      0,\n      6,\n      0,\n      "MODEL"\n    ],\n    [\n      3,\n      6,\n      0,\n      9,\n      0,\n      "MODEL"\n    ],\n    [\n      4,\n      3,\n      0,\n      7,\n      0,\n      "CLIP"\n    ],\n    [\n      5,\n      7,\n      0,\n      9,\n      1,\n      "CONDITIONING"\n    ],\n    [\n      6,\n      10,\n      0,\n      13,\n      0,\n      "NOISE"\n    ],\n    [\n      7,\n      9,\n      0,\n      13,\n      1,\n      "GUIDER"\n    ],\n    [\n      8,\n      11,\n      0,\n      13,\n      2,\n      "SAMPLER"\n    ],\n    [\n      9,\n      6,\n      0,\n      12,\n      0,\n      "MODEL"\n    ],\n    [\n      10,\n      12,\n      0,\n      13,\n      3,\n      "SIGMAS"\n    ],\n    [\n      11,\n      8,\n      0,\n      13,\n      4,\n      "LATENT"\n    ],\n    [\n      12,\n      13,\n      0,\n      14,\n      0,\n      "LATENT"\n    ],\n    [\n      13,\n      5,\n      0,\n      14,\n      1,\n      "MINIMAX_H3_PIXEL_HEAD"\n    ],\n    [\n      14,\n      6,\n      1,\n      14,\n      2,\n      "MINIMAX_H3_TAP"\n    ]\n  ],\n  "version": 0.4\n}', encoding="utf-8")
print("Google Drive 挂载成功，ComfyUI-MiniMaxH3-PixelHead 插件已同步至:", CUSTOM_NODE_DRIVE)

for sub, fn in [
    ("models/diffusion_models", "minimax_h3_fl2va_pruned_w6a8.safetensors"),
    ("models/text_encoders", "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("models/vae", "minimax_h3_video_vae_int8_convrot.safetensors"),
]:
    p = DRIVE_HUB / sub / fn
    st = f"✅ 网盘已就绪 ({p.stat().st_size/1e9:.2f} GB)" if p.exists() else "⬇️ 将在单元格 2 自动高速补齐"
    print(f"   - {sub}/{fn}: {st}")

In [ ]:
# @title 2. 升级最新 ComfyUI 引擎并将网盘已有的 MiniMax-H3 三件套预热至本地 NVMe SSD
import json, os, shutil, struct, subprocess, sys, time
from pathlib import Path

t0 = time.time()
DRIVE_HUB = Path("/content/drive/MyDrive/ComfyUI_Hub")
COMFY_DIR = Path("/content/ComfyUI")
LOCAL_MODELS = Path("/content/local_models")
LOCAL_MODELS.mkdir(parents=True, exist_ok=True)

# @markdown ### 主模型选择（默认直接使用你网盘已有的 14.65GB 8-Step PDD Turbo 8-Step PDD Turbo，也可切官方 pruned_w6a8）：
DIT_CHOICE = "minimax_h3_fl2va_8step_or_base"  # @param ["minimax_h3_fl2va_8step_or_base", "official_ref2va_pruned_w6a8"]

def get_hf_headers() -> list[str]:
    tok = os.environ.get("HF_TOKEN", "").strip()
    if not tok:
        try:
            from google.colab import userdata
            tok = (userdata.get("HF_TOKEN") or "").strip()
        except Exception:
            tok = ""
    return [f"--header=Authorization: Bearer {tok}"] if tok else []


def ensure_fresh_comfyui(required_rel_file: str) -> None:
    req_path = COMFY_DIR / required_rel_file
    if not COMFY_DIR.exists():
        # Prefer extracting an existing Hub tarball if available, then fast-forward
        candidates = sorted(DRIVE_HUB.glob("ComfyUI_*_bundle.tar.gz"))
        if candidates:
            print(f"从 Drive 极速解压 ComfyUI 基础包: {candidates[0].name} ...")
            subprocess.run(["tar", "-xzf", str(candidates[0]), "-C", "/content/"], check=True)
        else:
            print("📥 克隆官方最新 ComfyUI...")
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)], check=True)
    if not req_path.exists():
        print(f"升级 ComfyUI 至最新主干以启用 `{required_rel_file}`...")
        subprocess.run(["git", "-C", str(COMFY_DIR), "fetch", "--depth", "1", "origin", "master"], check=True)
        subprocess.run(["git", "-C", str(COMFY_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    print("安装/校验 ComfyUI 与评测依赖 (lpips, pyarrow, safetensors)...")
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", str(COMFY_DIR / "requirements.txt")], check=False)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "lpips", "pyarrow", "safetensors", "huggingface_hub", "einops"], check=True)
    if shutil.which("aria2c") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], check=False)


def is_valid_safetensors(p: Path, min_mb: float = 5.0) -> bool:
    if not p.exists() or p.stat().st_size < min_mb * 1024 * 1024:
        return False
    try:
        with open(p, "rb") as f:
            n = struct.unpack("<Q", f.read(8))[0]
            if n <= 0 or n > 100 * 1024 * 1024:
                return False
            hdr = json.loads(f.read(n))
        max_end = max((v["data_offsets"][1] for k, v in hdr.items() if k != "__metadata__"), default=0)
        return (8 + n + max_end) == p.stat().st_size
    except Exception:
        return False


def stage_model(rel_sub: str, fname: str, hf_urls: list[str] | str, min_mb: float, save_to_drive: bool = True, alt_drive_names: list[str] = []) -> Path:
    if isinstance(hf_urls, str):
        hf_urls = [hf_urls]
    local_p = LOCAL_MODELS / rel_sub / fname
    local_p.parent.mkdir(parents=True, exist_ok=True)
    if is_valid_safetensors(local_p, min_mb):
        print(f"  NVMe 已就绪: {fname} ({local_p.stat().st_size/1e9:.2f} GB)")
        return local_p

    drive_dir = DRIVE_HUB / "models" / rel_sub
    drive_dir.mkdir(parents=True, exist_ok=True)
    for cand_name in [fname] + list(alt_drive_names):
        cand_p = drive_dir / cand_name
        if is_valid_safetensors(cand_p, min_mb):
            dst_p = LOCAL_MODELS / rel_sub / cand_name
            if not is_valid_safetensors(dst_p, min_mb):
                print(f"  从 Drive 顺序预热至本地 NVMe: {cand_name} ({cand_p.stat().st_size/1e9:.2f} GB) ...")
                shutil.copyfile(cand_p, dst_p)
            return dst_p

    hdr_args = get_hf_headers()
    last_err = None
    for url in hf_urls:
        try:
            print(f"  本地 NVMe 高速直拉: {fname} <- {url.split('/resolve/')[0]} ...")
            tmp_dl = local_p.with_suffix(".dl")
            cmd = [
                "aria2c", "--console-log-level=warn", "--file-allocation=none",
                "-c", "-x", "16", "-s", "16", "-k", "1M",
                *hdr_args,
                url, "-d", str(local_p.parent), "-o", tmp_dl.name,
            ]
            subprocess.run(cmd, check=True)
            os.replace(tmp_dl, local_p)
            assert is_valid_safetensors(local_p, min_mb), f"下载文件头校验失败: {local_p}"
            if save_to_drive:
                shutil.copyfile(local_p, drive_dir / fname)
            return local_p
        except Exception as e:
            last_err = e
            print(f"  ⚠源 {url} 拉取失败，尝试备用源: {e}")
    raise RuntimeError(f"无法获取模型 {fname}: {last_err}")

ensure_fresh_comfyui("comfy/ldm/minimax/model.py")

if DIT_CHOICE == "minimax_h3_fl2va_8step_or_base":
    DIT_PATH = stage_model(
        "diffusion_models",
        "minimax_h3_fl2va_pruned_w6a8.safetensors",
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_w6a8.safetensors",
        min_mb=13000,
    )
else:
    DIT_PATH = stage_model(
        "diffusion_models",
        "minimax_h3_ref2va_pruned_w6a8.safetensors",
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_w6a8.safetensors",
        min_mb=14000,
    )

# 仅在单元格 3 离线缓存 8 条跨领域条件向量与 token_tags 时加载一次，随后彻底从显存与内存卸载！
TE_PATH = stage_model(
    "text_encoders",
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    min_mb=13000,
)

# 官方原版 Teacher Video VAE: minimax_h3_video_vae_int8_convrot.safetensors（含 3D Causal Encoder + 36层 ViT3D Decoder）
VAE_PATH = stage_model(
    "vae",
    "minimax_h3_video_vae_int8_convrot.safetensors",
    "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors",
    min_mb=2400,
)

for p in (str(COMFY_DIR), "/content"):
    if p not in sys.path:
        sys.path.insert(0, p)

import folder_paths
for k in ("diffusion_models", "text_encoders", "vae"):
    folder_paths.add_model_folder_path(k, str(LOCAL_MODELS / k))

print(f"\nMiniMax-H3 环境与 NVMe 模型预热完成。耗时: {time.time() - t0:.1f} 秒")

In [ ]:
# @title 3. 构建 22帧 (T_lat=7, 跨越双周期 1f/4f 边界!) 动态视频集并离线预编码（随后卸载 15.7GB 32B 编码器）
import gc, io, json, math, random, time, urllib.parse, urllib.request
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq
import torch
from PIL import Image, ImageDraw, ImageFont
import comfy.model_management, comfy.sd, comfy.utils

CACHE_ROOT = Path("/content/h3_ph_cache")
LAT_DIR, COND_DIR = CACHE_ROOT / "latents", CACHE_ROOT / "conds"
MANIFEST_PATH = CACHE_ROOT / "manifest.json"
for d in (LAT_DIR, COND_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 280 段 22 帧 (17*1 + 5 = 22 帧 -> T_lat = 7: t=0[1f], t=1..4[4f], t=5[1f], t=6[4f]) = 6,160 帧全监督！
TOTAL_CLIPS = 280    # @param {type:"integer"}
CLIP_W = 384         # @param {type:"integer"}  # H_lat=16, W_lat=24 -> DiT (1,2,2) patch 网格 7x8x12 = 672 tokens
CLIP_H = 256         # @param {type:"integer"}
CLIP_FRAMES = 22     # @param {type:"integer"}  # 严格对齐 MiniMax-H3 17k+5 网格 (k=1 -> 22 frames, T_lat=7)

def center_crop_resize(im: Image.Image, size: int) -> Image.Image:
    im = im.convert("RGB")
    w, h = im.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    return im.crop((left, top, left + side, top + side)).resize((size, size), Image.Resampling.LANCZOS)


def make_synthetic_cel_manga_typo(idx: int, size: int, title_banner: str) -> tuple[Image.Image, str]:
    rng = random.Random(20261122 + idx)
    sub_kind = idx % 3
    if sub_kind == 0:
        bg_c = (rng.randint(215, 250), rng.randint(220, 250), rng.randint(230, 255))
        im = Image.new("RGB", (size, size), bg_c)
        dr = ImageDraw.Draw(im)
        for _ in range(14):
            x0, y0 = rng.randint(0, size - 120), rng.randint(0, size - 120)
            x1, y1 = x0 + rng.randint(80, 360), y0 + rng.randint(80, 360)
            fill = (rng.randint(30, 245), rng.randint(30, 245), rng.randint(30, 245))
            outline = (18, 18, 24)
            w_line = rng.choice([2, 3, 4])
            if rng.random() < 0.5:
                dr.ellipse([x0, y0, x1, y1], fill=fill, outline=outline, width=w_line)
            else:
                dr.rounded_rectangle([x0, y0, x1, y1], radius=18, fill=fill, outline=outline, width=w_line)
        return im, "anime_cel"
    elif sub_kind == 1:
        arr = np.full((size, size, 3), 248, dtype=np.uint8)
        yy, xx = np.mgrid[0:size, 0:size]
        period = rng.choice([4, 6, 8])
        dots = (((xx % period == 0) & (yy % period == 0))).astype(np.uint8)
        arr[size // 6 : size // 2, size // 6 : size * 5 // 6] -= dots[size // 6 : size // 2, size // 6 : size * 5 // 6, None] * 170
        im = Image.fromarray(arr, mode="RGB")
        dr = ImageDraw.Draw(im)
        for _ in range(35):
            p0 = (rng.randint(0, size), rng.randint(0, size))
            p1 = (rng.randint(0, size), rng.randint(0, size))
            dr.line([p0, p1], fill=(12, 12, 12), width=rng.choice([1, 2, 3]))
        return im, "manga_line"
    else:
        dark = rng.random() < 0.5
        bg = (18, 20, 28) if dark else (246, 244, 238)
        fg = (242, 245, 250) if dark else (20, 22, 28)
        accent = (rng.randint(180, 255), rng.randint(60, 190), rng.randint(50, 220))
        im = Image.new("RGB", (size, size), bg)
        dr = ImageDraw.Draw(im)
        dr.rectangle([36, 36, size - 36, size - 36], outline=accent, width=4)
        sample_lines = [
            title_banner,
            "FastFrames Zero-VAE Readout 2026",
            "0123456789  PSNR 35.8 dB  5.0 ms",
            "Sharp Edges · Clean Cel · Fine Detail",
            "abcdefghijklmnopqrstuvwxyz ABCDEFG",
        ]
        y_cur = 75
        for li, text in enumerate(sample_lines):
            fsize = [40, 28, 24, 20, 16][li % 5]
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", fsize)
            except Exception:
                font = ImageFont.load_default()
            dr.text((60, y_cur), text, fill=fg if li % 2 == 0 else accent, font=font)
            y_cur += fsize + 36
        return im, "typography"


def fetch_hf_parquet_pil(dataset_id: str, max_count: int, min_side: int = 384) -> list[Image.Image]:
    pil_list = []
    shard_urls = []
    try:
        pq_api = f"https://huggingface.co/api/datasets/{dataset_id}/parquet"
        req = urllib.request.Request(pq_api, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=25) as r:
            pq_tree = json.loads(r.read().decode("utf-8"))
        for cfg_map in pq_tree.values():
            if isinstance(cfg_map, dict):
                for split_list in cfg_map.values():
                    if isinstance(split_list, list):
                        shard_urls.extend(split_list)
    except Exception:
        pass
    if not shard_urls:
        api_url = f"https://huggingface.co/api/datasets/{dataset_id}"
        req = urllib.request.Request(api_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=25) as r:
            info = json.loads(r.read().decode("utf-8"))
        for s in info.get("siblings", []):
            rf = s.get("rfilename", "")
            if rf.endswith(".parquet"):
                shard_urls.append(f"https://huggingface.co/datasets/{dataset_id}/resolve/main/{urllib.parse.quote(rf)}")

    for idx_sh, sh_url in enumerate(shard_urls):
        if len(pil_list) >= max_count:
            break
        local_pq = CACHE_ROOT / f"_tmp_{dataset_id.replace('/', '_')}_{idx_sh:03d}.parquet"
        if not local_pq.exists():
            print(f"  拉取公开无门控校准分片 [{idx_sh}]: {dataset_id} ...")
            urllib.request.urlretrieve(sh_url, str(local_pq))
        table = pq.read_table(local_pq)
        img_col = next((c for c in table.column_names if "image" in c.lower()), None)
        if img_col is None:
            local_pq.unlink(missing_ok=True)
            continue
        for row_val in table[img_col].to_pylist():
            if len(pil_list) >= max_count:
                break
            raw_bytes = row_val.get("bytes") if isinstance(row_val, dict) else None
            if not raw_bytes:
                continue
            try:
                im = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
                if min(im.size) < min_side:
                    continue
                pil_list.append(im)
            except Exception:
                continue
        local_pq.unlink(missing_ok=True)
    return pil_list


def synthesize_motion_clip(src_pil: Image.Image, out_w: int, out_h: int, num_frames: int, idx: int) -> tuple[torch.Tensor, str]:
    """Generate temporally smooth `[F, H, W, 3]` float32 clip in `[0, 1]` covering 4 motion regimes:
      0: Smooth subpixel Ken Burns camera pan/zoom (tests subpixel temporal continuity)
      1: Multi-layer foreground object parallax over panning scene (tests sharp motion occlusion)
      2: Dynamic 2D cel / kinetic typography sweep (tests crisp moving edges & text)
      3: Static tripod lock-off (15% — enforces zero temporal boiling/flicker on static scenes!)
    """
    rng = random.Random(77000 + idx)
    mode = 3 if (idx % 7 == 0) else (idx % 3)

    if mode in (0, 3):
        scale_pad = 1.32 if mode == 0 else 1.05
        big_w, big_h = int(out_w * scale_pad), int(out_h * scale_pad)
        canvas = src_pil.convert("RGB").resize((big_w, big_h), Image.Resampling.LANCZOS)
        t_nchw = torch.from_numpy(np.array(canvas)).float().div_(255.0).permute(2, 0, 1).unsqueeze(0)  # [1,3,Hb,Wb]
        if mode == 3:
            y0, x0 = (big_h - out_h) // 2, (big_w - out_w) // 2
            crop = t_nchw[:, :, y0 : y0 + out_h, x0 : x0 + out_w]
            return crop.repeat(num_frames, 1, 1, 1).permute(0, 2, 3, 1).contiguous(), "static_lockoff"

        # Subpixel affine pan + subtle zoom via grid_sample
        dx0, dy0 = rng.uniform(-0.18, 0.18), rng.uniform(-0.18, 0.18)
        dx1, dy1 = -dx0 * rng.uniform(0.6, 1.0), -dy0 * rng.uniform(0.6, 1.0)
        z0, z1 = rng.uniform(0.76, 0.86), rng.uniform(0.76, 0.86)
        frames = []
        for fi in range(num_frames):
            alpha = fi / max(1, num_frames - 1)
            # Smooth cosine ease-in-out
            u = 0.5 - 0.5 * math.cos(math.pi * alpha)
            zx = z0 + (z1 - z0) * u
            tx = dx0 + (dx1 - dx0) * u
            ty = dy0 + (dy1 - dy0) * u
            theta = torch.tensor([[[zx, 0.0, tx], [0.0, zx, ty]]], dtype=torch.float32)
            grid = torch.nn.functional.affine_grid(theta, size=(1, 3, out_h, out_w), align_corners=False)
            fr = torch.nn.functional.grid_sample(t_nchw, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
            frames.append(fr[0])
        return torch.stack(frames, dim=0).permute(0, 2, 3, 1).clamp_(0.0, 1.0).contiguous(), "camera_pan_zoom"

    elif mode == 1:
        # Multi-layer parallax: slow background pan + crisp moving foreground badge/character silhouette
        big_w, big_h = int(out_w * 1.25), int(out_h * 1.25)
        bg_pil = src_pil.convert("RGB").resize((big_w, big_h), Image.Resampling.LANCZOS)
        bg_t = torch.from_numpy(np.array(bg_pil)).float().div_(255.0)
        fg_r = min(out_h, out_w) // rng.choice([4, 5])
        fg_col = torch.tensor([rng.uniform(0.15, 0.95), rng.uniform(0.15, 0.95), rng.uniform(0.15, 0.95)])
        border_col = torch.tensor([0.07, 0.07, 0.09])
        yy, xx = torch.meshgrid(torch.arange(out_h, dtype=torch.float32), torch.arange(out_w, dtype=torch.float32), indexing="ij")
        vx0, vy0 = rng.uniform(0.25, 0.75) * out_w, rng.uniform(0.3, 0.7) * out_h
        vx1, vy1 = rng.uniform(0.25, 0.75) * out_w, rng.uniform(0.3, 0.7) * out_h
        frames = []
        for fi in range(num_frames):
            a = fi / max(1, num_frames - 1)
            bx = int((big_w - out_w) * a)
            by = int((big_h - out_h) * (1.0 - a))
            fr = bg_t[by : by + out_h, bx : bx + out_w].clone()
            cx = vx0 + (vx1 - vx0) * a
            cy = vy0 + (vy1 - vy0) * a
            dist = torch.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
            mask_in = dist <= (fg_r - 3.0)
            mask_bd = (dist > (fg_r - 3.0)) & (dist <= fg_r)
            fr[mask_in] = fg_col
            fr[mask_bd] = border_col
            frames.append(fr)
        return torch.stack(frames, dim=0).clamp_(0.0, 1.0).contiguous(), "parallax_occlusion"

    else:
        # Kinetic 2D cel & sharp typography scroll
        dark = rng.random() < 0.5
        bg = (20, 22, 30) if dark else (244, 242, 236)
        fg = (242, 245, 250) if dark else (22, 24, 30)
        acc = (rng.randint(170, 250), rng.randint(60, 190), rng.randint(60, 220))
        frames = []
        for fi in range(num_frames):
            im = Image.new("RGB", (out_w, out_h), bg)
            dr = ImageDraw.Draw(im)
            shift_x = int((fi / max(1, num_frames - 1) - 0.5) * (out_w * 0.35))
            dr.rounded_rectangle([out_w // 6 + shift_x, out_h // 5, out_w * 5 // 6 + shift_x, out_h * 4 // 5], radius=16, outline=acc, width=3)
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 24)
            except Exception:
                font = ImageFont.load_default()
            dr.text((out_w // 5 + shift_x, out_h // 2 - 14), "FASTFRAMES 3D VIDEO", fill=fg, font=font)
            frames.append(torch.from_numpy(np.array(im)).float().div_(255.0))
        return torch.stack(frames, dim=0).contiguous(), "kinetic_cel_typo"

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"检测到已完成预编码的本地 H3 视频潜空间缓存 ({len(manifest)} 段视频)，直接复用！")
else:
    t_c = time.time()
    print(f"[1/3] 拉取高清源图合成 {TOTAL_CLIPS} 段 22 帧多模态运动视频 ({CLIP_W}x{CLIP_H}x{CLIP_FRAMES})...")
    src_imgs = fetch_hf_parquet_pil("huggan/wikiart", TOTAL_CLIPS // 2, min_side=480)
    src_imgs += fetch_hf_parquet_pil("detection-datasets/coco", TOTAL_CLIPS - len(src_imgs), min_side=480)

    print(f"[2/3] 使用官方 Teacher Video VAE ({VAE_PATH.name}) 编码 {TOTAL_CLIPS} 段 24ch 潜变量 (T_lat=7)...")
    vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(VAE_PATH)))
    rows = []
    for idx_c in range(TOTAL_CLIPS):
        base_im = src_imgs[idx_c % len(src_imgs)]
        frames_fhwc, motion_tag = synthesize_motion_clip(base_im, CLIP_W, CLIP_H, CLIP_FRAMES, idx_c)
        with torch.no_grad():
            z = vae.encode(frames_fhwc)  # [1, 24, 7, 16, 24]
        lat_fn = f"clip_{idx_c:04d}.pt"
        torch.save(z.to(torch.bfloat16).cpu(), LAT_DIR / lat_fn)
        rows.append({"id": idx_c, "latent": lat_fn, "domain": motion_tag})
        if (idx_c + 1) % 40 == 0 or (idx_c + 1) == TOTAL_CLIPS:
            print(f"    已编码 H3 Video Latent: {idx_c + 1}/{TOTAL_CLIPS} | shape={tuple(z.shape)}")

    del vae, src_imgs
    comfy.model_management.cleanup_models_gc(); gc.collect(); torch.cuda.empty_cache()

    rng_split = random.Random(42)
    idxs = list(range(len(rows))); rng_split.shuffle(idxs)
    test_set = set(idxs[:24])
    for i, r in enumerate(rows):
        r["split"] = "test" if i in test_set else "train"

    print("[3/3] 加载 Qwen3-VL-32B (CLIPType.MINIMAX) 编码文本嵌入与 minimax_token_tags，随后彻底清空 15.7GB 显存...")
    PROMPTS = {
        "camera_pan_zoom": ["Cinematic 24fps shot with smooth camera push-in and pan across a detailed artistic scene, crisp lighting. Audio: soft ambient wind."],
        "parallax_occlusion": ["Smooth foreground motion across a panning background with sharp edges and natural movement. Audio: subtle room tone."],
        "kinetic_cel_typo": ["Crisp 2D anime cel animation with bold linework and clean moving typography. Audio: gentle synth chime."],
        "static_lockoff": ["Steady tripod shot with crisp stationary textures and calm natural lighting. Audio: quiet room tone."],
    }
    clip = comfy.sd.load_clip(ckpt_paths=[str(TE_PATH)], embedding_directory=None, clip_type=comfy.sd.CLIPType.MINIMAX)
    cond_map, cid_counter = {}, 0
    for dom, plist in PROMPTS.items():
        cond_map[dom] = []
        for ptxt in plist:
            cond = clip.encode_from_tokens_scheduled(clip.tokenize(ptxt))
            payload_c = {
                "cross_attn": cond[0][0].detach().to(torch.bfloat16).cpu(),
                "minimax_token_tags": cond[0][1].get("minimax_token_tags").cpu() if cond[0][1].get("minimax_token_tags") is not None else None,
            }
            cfn = f"cond_{cid_counter:03d}.pt"
            torch.save(payload_c, COND_DIR / cfn)
            cond_map[dom].append(cfn)
            cid_counter += 1
    for r in rows:
        opts = cond_map.get(r["domain"], cond_map["camera_pan_zoom"])
        r["cond"] = opts[r["id"] % len(opts)]

    del clip
    comfy.model_management.unload_all_models(); comfy.model_management.soft_empty_cache(); gc.collect(); torch.cuda.empty_cache()
    # 删除本地 NVMe 上的 15.7GB 文本编码器临时副本，释放磁盘空间给训练检查点
    try:
        TE_PATH.unlink(missing_ok=True)
    except Exception:
        pass
    MANIFEST_PATH.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    manifest = rows
    print(f"MiniMax-H3 预编码完成且 32B 编码器已彻底释放！耗时 {time.time() - t_c:.1f}s | 空闲显存: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

In [ ]:
# @title 4. P0 跨分辨率等变门禁 + 1800 Video Rollouts 过完备 3D 正式训练（自适应 PDD 时间表 + 全画幅混合监督）
from collections import deque
import gc, importlib.util, json, math, random, shutil, time
from pathlib import Path
import torch
from safetensors.torch import save_file
import comfy.model_management, comfy.model_sampling, comfy.sd, comfy.utils

ARCH_VERSION = "minimax_h3_v2_overcomplete"
LOCAL_MODELS = Path("/content/local_models")
if "DIT_PATH" not in globals():
    cands = list((LOCAL_MODELS / "diffusion_models").glob("*.safetensors"))
    DIT_PATH = cands[0]
if "VAE_PATH" not in globals():
    VAE_PATH = LOCAL_MODELS / "vae/minimax_h3_video_vae_int8_convrot.safetensors"

spec = importlib.util.spec_from_file_location("minimax_h3_pixel_head", "/content/ComfyUI-MiniMaxH3-PixelHead/minimax_h3_pixel_head.py")
ph_mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(ph_mod)
MiniMaxH3VideoPixelHead = ph_mod.MiniMaxH3VideoPixelHead
h3_expected_frames = ph_mod.h3_expected_frames
save_minimax_h3_pixel_head = ph_mod.save_minimax_h3_pixel_head
load_minimax_h3_pixel_head = ph_mod.load_minimax_h3_pixel_head

# 1800 Video Rollouts × 16 步更新 = 28,800 步梯度更新（L4 约 32~40 分钟）
TOTAL_ROLLOUTS = 1800        # @param {type:"integer"}
REPLAY_BUFFER_SIZE = 36      # @param {type:"integer"}
UPDATES_PER_ROLLOUT = 16     # @param {type:"integer"}
HEAD_WIDTH = 768             # @param [576, 768] {type:"raw"}
LEARNING_RATE = 3e-4         # @param {type:"number"}
WARMUP_STEPS = 200           # @param {type:"integer"}
SAVE_EVERY_ROLLOUTS = 200    # @param {type:"integer"}
RESUME_IF_EXISTS = True      # @param {type:"boolean"}

DEVICE = torch.device("cuda")
CKPT_DRIVE = Path("/content/drive/MyDrive/FastFrames_MiniMaxH3/checkpoints")
PIXEL_HEADS_DRIVE = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads")
VAE_DRIVE = Path("/content/drive/MyDrive/ComfyUI_Hub/models/vae")

manifest = json.loads(Path("/content/h3_ph_cache/manifest.json").read_text(encoding="utf-8"))
train_rows = [r for r in manifest if r["split"] == "train"]
test_rows = [r for r in manifest if r["split"] == "test"]
random.Random(20261122).shuffle(train_rows)

# 自动检测并归档旧版 v1 断点
latest_ckpt = CKPT_DRIVE / "latest_state.pt"
valid_resume_state = None
if RESUME_IF_EXISTS and latest_ckpt.exists():
    try:
        peek_st = torch.load(latest_ckpt, map_location="cpu", weights_only=False)
        if peek_st.get("arch_version") == ARCH_VERSION and peek_st.get("head_width") == HEAD_WIDTH:
            valid_resume_state = peek_st
        else:
            bak_p = CKPT_DRIVE / "latest_state_v1_archived.pt"
            shutil.move(str(latest_ckpt), str(bak_p))
            print(f"⚠检测到旧版 v1 断点 (arch={peek_st.get('arch_version', 'v1_legacy')})，已自动归档至 {bak_p.name} 并从头启动 v2 高清架构训练！")
    except Exception as e:
        print(f"⚠旧断点读取异常 ({e})，将从头启动 v2 训练。")

comfy.model_management.unload_all_models(); gc.collect(); torch.cuda.empty_cache()
print("加载冻结的 MiniMax-H3 (Base & 8-Step PDD Turbo) DiT (配置 video_shift=12.0 / audio_shift=3.0) 与官方 Video VAE...")
patcher_h3 = comfy.sd.load_diffusion_model(str(DIT_PATH))


def apply_h3_sigma_shift(model_patcher, shift_video: float = 12.0, shift_audio: float = 3.0):
    m = model_patcher.clone()
    ms_base = comfy.model_sampling.ModelSamplingAV

    class ModelSamplingAdvanced(ms_base, comfy.model_sampling.CONST):
        pass

    orig = m.get_model_object("model_sampling")
    ms_new = ModelSamplingAdvanced(m.model.model_config)
    ms_new.set_parameters(shift=shift_video, audio_shift=shift_audio)
    if hasattr(orig, "noise_scale"):
        ms_new.set_noise_scale(orig.noise_scale)
    m.add_object_patch("model_sampling", ms_new)
    to = m.model_options["transformer_options"] = m.model_options.get("transformer_options", {}).copy()
    to["minimax_h3_sigma_shift_video"] = shift_video
    to["minimax_h3_sigma_shift_audio"] = shift_audio
    return m


patcher_h3 = apply_h3_sigma_shift(patcher_h3, 12.0, 3.0)
teacher_vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(VAE_PATH)))

# 自动检测底模是否为 PDD 分段分布蒸馏模型 (如 8-Step PDD Turbo 8-Step Turbo, n_pdd = 8 > 1)
_fl_vo = patcher_h3.model.diffusion_model.final_layer.video_out
N_PDD_HEADS = int(_fl_vo.weight.shape[0] // _fl_vo.out_features)
_dn_low = DIT_PATH.name.lower()
IS_PDD_MODEL = (N_PDD_HEADS > 1) or ("turbo" in _dn_low) or ("8step" in _dn_low)
print(f"自动识别底模类型: {'8-Step Turbo / PDD 加速蒸馏模型 (聚焦 4~16步真实末步区间 [0.444, 0.800])' if IS_PDD_MODEL else '标准流匹配 Base/Ref2VA 模型 (覆盖 [0.180, 0.800])'}")

TURBO8_SIGMA_ANCHOR = 0.63158   # shift=12.0 下 8 步精确末步 sigma
TURBO12_SIGMA_ANCHOR = 0.52174  # shift=12.0 下 12 步精确末步 sigma
PDD_MAIN_JITTER = (0.550, 0.710)
PDD_WIDE_RANGE = (0.444, 0.800) # 覆盖 4步(0.80) ~ 16步(0.444) 全部真实 PDD 末步
BASE_SIGMA_RANGE = (0.180, 0.480)


class MiniMaxH3RolloutEngine:
    def __init__(self, patcher, vae, cache_dir: Path):
        self.patcher = patcher
        self.vae = vae
        self.lat_dir = cache_dir / "latents"
        self.cond_dir = cache_dir / "conds"
        self.cond_mem = {}

    def _get_cond_obj(self, fn: str) -> dict:
        if fn not in self.cond_mem:
            self.cond_mem[fn] = torch.load(self.cond_dir / fn, map_location="cpu", weights_only=True)
        return self.cond_mem[fn]

    @torch.no_grad()
    def rollout(self, row: dict, sigma: float, seed: int):
        comfy.model_management.load_models_gpu([self.patcher])
        bmodel = self.patcher.model

        x0_vid = torch.load(self.lat_dir / row["latent"], map_location=DEVICE, weights_only=True).float()
        _, _, t_lat, h_lat, w_lat = x0_vid.shape
        f_pix = h3_expected_frames(t_lat)
        t_aud = round(f_pix * (5.0 / 3.0))

        gen = torch.Generator(device=DEVICE).manual_seed(int(seed))
        xt_vid = (1.0 - sigma) * x0_vid + sigma * torch.randn(x0_vid.shape, generator=gen, device=DEVICE)
        xt_aud = sigma * torch.randn((1, 32, 2, t_aud), generator=gen, device=DEVICE, dtype=torch.float32)

        packed_in, latent_shapes = comfy.utils.pack_latents([xt_vid, xt_aud])
        bmodel.latent_shapes = latent_shapes
        packed_scaled = bmodel.process_latent_in(packed_in)

        cobj = self._get_cond_obj(row["cond"])
        ec = bmodel.extra_conds(
            cross_attn=cobj["cross_attn"],
            minimax_token_tags=cobj["minimax_token_tags"],
            latent_shapes=latent_shapes,
            seed=int(seed),
            device=DEVICE,
        )
        kw_conds = {k: v.cond for k, v in ec.items()}
        c_crossattn = kw_conds.pop("c_crossattn", None)

        sig_t = torch.tensor([sigma], device=DEVICE, dtype=torch.float32)
        topts = {
            "sample_sigmas": torch.tensor([sigma, 0.0], device=DEVICE, dtype=torch.float32),
            "minimax_h3_sigma_shift_video": 12.0,
            "minimax_h3_sigma_shift_audio": 3.0,
        }

        captured = {}

        def _pre_fl_hook(fl, inputs):
            if "f" not in captured and len(inputs) >= 3:
                h, t_emb, video_seg = inputs[0], inputs[1], inputs[2]
                va, vb, row_idx = video_seg
                shift, scale = fl.adaln_proj(t_emb)
                sc = scale[row_idx].to(scale.dtype)
                sh = shift[row_idx].to(shift.dtype)
                captured["f"] = (fl.norm(h[va:vb]) * (1.0 + sc) + sh).detach()

        handle = bmodel.diffusion_model.final_layer.register_forward_pre_hook(_pre_fl_hook)
        try:
            den_packed = bmodel.apply_model(
                packed_scaled, sig_t, c_crossattn=c_crossattn, transformer_options=topts, **kw_conds
            ).float()
        finally:
            handle.remove()

        den_unscaled = bmodel.process_latent_out(den_packed)
        x0_hat_vid = comfy.utils.unpack_latents(den_unscaled, latent_shapes)[0]

        dec = self.vae.decode(x0_hat_vid.cpu())[..., :3]
        vid_fhwc = dec[0] if dec.ndim == 5 else dec
        tgt_cthw = (vid_fhwc.permute(3, 0, 1, 2).float() * 2.0 - 1.0).clamp_(-1.0, 1.0)

        gh, gw = (h_lat + 1) // 2, (w_lat + 1) // 2
        f_grid = captured["f"].float().view(t_lat, gh, gw, -1)
        return (
            f_grid.to(torch.bfloat16).cpu(),
            x0_hat_vid[0].to(torch.bfloat16).cpu(),
            tgt_cthw.to(torch.bfloat16).cpu(),
        )


engine = MiniMaxH3RolloutEngine(patcher_h3, teacher_vae, Path("/content/h3_ph_cache"))


def fastframes_video_loss(pred: torch.Tensor, tgt: torch.Tensor) -> tuple[torch.Tensor, float]:
    diff = (pred - tgt).abs()
    l1 = diff.mean()
    mse = ((pred - tgt) ** 2).mean()
    gh = (pred.diff(dim=-2) - tgt.diff(dim=-2)).abs().mean()
    gw = (pred.diff(dim=-1) - tgt.diff(dim=-1)).abs().mean()
    if pred.shape[2] > 1:
        dt_pred = pred[:, :, 1:] - pred[:, :, :-1]
        dt_tgt = tgt[:, :, 1:] - tgt[:, :, :-1]
        temp_l1 = (dt_pred - dt_tgt).abs().mean()
        motion_w = dt_tgt.abs().mean(dim=1, keepdim=True).clamp(0.0, 1.0)
        motion_l1 = (diff[:, :, 1:] * motion_w).mean()
    else:
        temp_l1 = pred.new_zeros(())
        motion_l1 = pred.new_zeros(())
    total = 0.5 * l1 + 5.0 * mse + 0.05 * (gh + gw) + 0.05 * temp_l1 + 0.5 * motion_l1
    return total, -10.0 * math.log10(max(1e-10, float(mse.detach().item()) * 0.25))


def strip_video_margin(vid: torch.Tensor, mar_px: tuple[int, int, int, int]) -> torch.Tensor:
    mt, mb, ml, mr = mar_px
    h, w = vid.shape[-2], vid.shape[-1]
    return vid[..., mt : h - mb, ml : w - mr]


if valid_resume_state is None:
    print("\n[Stage 4A] 运行 P0 3D 空间平移等变精确校验 + 8x12 全画幅收敛门禁...")
    smoke_head = MiniMaxH3VideoPixelHead(width=384).to(DEVICE)
    assert smoke_head.out_proj.in_features >= smoke_head.out_proj.out_features, "输出头存在欠完备线性秩瓶颈！"
    # 1) 0.1秒确定性数学检验：校验异形分辨率裁剪内部严格平移等变 (<1e-3) 与 decode_chunked 零接缝等价 (<1e-4)
    with torch.no_grad():
        smoke_head.eval()
        tf = torch.randn(1, 17, 4, 6, 5376, device=DEVICE, dtype=torch.float32)
        tx = torch.randn(1, 24, 17, 8, 12, device=DEVICE, dtype=torch.float32)
        full_o = smoke_head(tf, tx).cpu()
        chk_o = smoke_head.decode_chunked(tf, tx, chunk_t=5)
        assert (full_o - chk_o).abs().max().item() < 1e-4, "decode_chunked 与 forward 不一致！"
        tf_b = torch.randn(1, 5, 14, 18, 5376, device=DEVICE, dtype=torch.float32)
        tx_b = torch.randn(1, 24, 5, 28, 36, device=DEVICE, dtype=torch.float32)
        ob_full = smoke_head(tf_b, tx_b)
        ob_sub = smoke_head(tf_b[:, :, 2:12, 3:15], tx_b[:, :, :, 4:24, 6:30])
        eq_err = float((ob_full[:, :, :, 6*32:8*32, 7*32:11*32] - ob_sub[:, :, :, 4*32:6*32, 4*32:8*32]).abs().max().item())
        assert eq_err < 1e-3, f"3D PixelShuffle 空间轴等变校验失败: err={eq_err}"
        del tf, tx, full_o, chk_o, tf_b, tx_b, ob_full, ob_sub
        smoke_head.train()

    # 2) 全画幅 8x12 直接训练 90 步（四角像素获得完整梯度覆盖，无边缘饥饿）
    smoke_items = []
    for i in range(2):
        f_g, x_g, t_g = engine.rollout(train_rows[i], TURBO8_SIGMA_ANCHOR, 1000 + i)
        assert f_g.shape[-1] == 5376 and x_g.shape[0] == 24 and t_g.shape[1] == 22, f"形状异常: {f_g.shape}, {x_g.shape}, {t_g.shape}"
        smoke_items.append((f_g.unsqueeze(0).to(DEVICE, torch.bfloat16), x_g.unsqueeze(0).to(DEVICE, torch.bfloat16), t_g.unsqueeze(0).to(DEVICE, torch.float32)))

    smoke_opt = torch.optim.AdamW(smoke_head.parameters(), lr=8e-4, betas=(0.9, 0.99), weight_decay=0.0)
    init_p, last_p = [], []
    for st in range(1, 91):
        f_s, x_s, t_s = smoke_items[st % 2]
        smoke_opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = smoke_head(f_s, x_s)
        loss, pv = fastframes_video_loss(pred, t_s)
        assert math.isfinite(float(loss.item())), "NaN/Inf Loss!"
        loss.backward(); torch.nn.utils.clip_grad_norm_(smoke_head.parameters(), 1.0); smoke_opt.step()
        if st <= 4:
            init_p.append(pv)
        if st > 82:
            last_p.append(pv)
    p0_s, p0_e = sum(init_p) / len(init_p), sum(last_p) / len(last_p)
    del smoke_head, smoke_opt, smoke_items; torch.cuda.empty_cache()
    print(f"  P0 MiniMax-H3 等变与全画幅门禁通过 (eq_err={eq_err:.2e})！90 步全画幅 PSNR: {p0_s:.2f} dB -> {p0_e:.2f} dB")
    assert p0_e >= 17.5 and (p0_e - p0_s) >= 5.0, f"全画幅门禁爬升不足: {p0_s:.2f} -> {p0_e:.2f} dB"

head = MiniMaxH3VideoPixelHead(width=HEAD_WIDTH).to(DEVICE)
n_params = sum(p.numel() for p in head.parameters())
print(f"\n[Stage 4B] 正式训练 MiniMaxH3VideoPixelHead v2: width={HEAD_WIDTH}, 参数量={n_params/1e6:.2f}M (~{n_params*4/(1024*1024):.1f} MiB FP32 / ~{n_params*2/(1024*1024):.1f} MiB BF16)")

total_grad_steps = TOTAL_ROLLOUTS * UPDATES_PER_ROLLOUT
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=1e-2, betas=(0.9, 0.99))


def set_step_lr(opt, cur_step: int) -> float:
    warm = min(1.0, (cur_step + 1) / max(1, WARMUP_STEPS))
    prog = min(1.0, cur_step / max(1, total_grad_steps))
    lr_now = LEARNING_RATE * warm * (0.08 + 0.92 * 0.5 * (1.0 + math.cos(math.pi * prog)))
    for pg in opt.param_groups:
        pg["lr"] = lr_now
    return lr_now


start_rollout, opt_step_count, best_eval_psnr, history = 0, 0, 0.0, []
if valid_resume_state is not None:
    head.load_state_dict(valid_resume_state["head"])
    optimizer.load_state_dict(valid_resume_state["optimizer"])
    start_rollout = int(valid_resume_state["rollout"])
    opt_step_count = int(valid_resume_state.get("opt_step_count", start_rollout * UPDATES_PER_ROLLOUT))
    best_eval_psnr = float(valid_resume_state.get("best_eval_psnr", 0.0))
    history = list(valid_resume_state.get("history", []))
    del valid_resume_state
    print(f"从 v2 断点恢复: Rollout {start_rollout}/{TOTAL_ROLLOUTS} (best={best_eval_psnr:.2f} dB)")

replay = deque(maxlen=REPLAY_BUFFER_SIZE)
rng = random.Random(20261122 + start_rollout)


def save_bf16_variant(src_fp32_path: Path, dst_bf16_path: Path) -> None:
    from safetensors import safe_open
    with safe_open(str(src_fp32_path), framework="pt", device="cpu") as fh:
        meta = dict(fh.metadata() or {})
        sd_bf16 = {}
        for k in fh.keys():
            t = fh.get_tensor(k)
            sd_bf16[k] = t.float().contiguous() if k.startswith("out_proj.") else t.to(torch.bfloat16).contiguous()
    meta["storage_precision"] = "bf16_trunk_fp32_out"
    save_file(sd_bf16, str(dst_bf16_path), metadata=meta)


@torch.no_grad()
def evaluate_heldout(n_eval: int = 12) -> dict:
    head.eval()
    ps_t, ps_s = [], []
    alt_sig = TURBO12_SIGMA_ANCHOR if IS_PDD_MODEL else 0.300
    for idx_e, row in enumerate(test_rows[: min(n_eval, len(test_rows))]):
        use_t = (idx_e % 2 == 0)
        sig = TURBO8_SIGMA_ANCHOR if use_t else alt_sig
        f_g, x_g, t_g = engine.rollout(row, sig, 880000 + idx_e)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = head(f_g.unsqueeze(0).to(DEVICE, torch.bfloat16), x_g.unsqueeze(0).to(DEVICE, torch.bfloat16))
        p01 = ((pred[0].float().cpu() + 1.0) * 0.5).clamp(0, 1)
        t01 = ((t_g.float() + 1.0) * 0.5).clamp(0, 1)
        db = -10.0 * math.log10(max(1e-10, float(((p01 - t01) ** 2).mean().item())))
        (ps_t if use_t else ps_s).append(db)
    head.train()
    all_p = ps_t + ps_s
    return {"mean_psnr": sum(all_p) / len(all_p), "turbo8_psnr": sum(ps_t) / max(1, len(ps_t)), "secondary_psnr": sum(ps_s) / max(1, len(ps_s))}


t_train = time.time()
run_loss, run_psnr, run_n = 0.0, 0.0, 0
for r_idx in range(start_rollout + 1, TOTAL_ROLLOUTS + 1):
    row = train_rows[(r_idx - 1) % len(train_rows)]
    u_mode = rng.random()
    if IS_PDD_MODEL:
        # PDD 模型 (8-Step PDD Turbo): 40% 精确锚定 8步末步 0.63158, 35% 7~10步微扰, 25% 4~16步全宽 PDD 区间
        if u_mode < 0.40:
            sigma = TURBO8_SIGMA_ANCHOR
        elif u_mode < 0.75:
            sigma = rng.uniform(*PDD_MAIN_JITTER)
        else:
            sigma = rng.uniform(*PDD_WIDE_RANGE)
    else:
        if u_mode < 0.50:
            sigma = TURBO8_SIGMA_ANCHOR if (r_idx % 4 == 1) else rng.uniform(*PDD_WIDE_RANGE)
        else:
            sigma = rng.uniform(*BASE_SIGMA_RANGE)

    replay.append(engine.rollout(row, sigma, 100000 + r_idx))
    n_upd = UPDATES_PER_ROLLOUT if len(replay) >= 6 else 4
    cur_lr = LEARNING_RATE
    for u_i in range(n_upd):
        cur_lr = set_step_lr(optimizer, opt_step_count)
        bf, bx, bt = replay[rng.randrange(len(replay))]
        _, gh, gw, _ = bf.shape

        if u_i % 4 != 3:
            # 75% 全画幅 8x12 (384x256x22f) 监督，零裁剪边界偏差
            f_sub = bf.unsqueeze(0).to(DEVICE, torch.bfloat16)
            x_sub = bx.unsqueeze(0).to(DEVICE, torch.bfloat16)
            t_sub = bt.unsqueeze(0).to(DEVICE, torch.float32)
            mar_px = (0, 0, 0, 0)
        else:
            # 25% 异形子窗口 (6x8) + 内部边缘剥离 1 token (32px)
            sh, sw = min(gh, 6), min(gw, 8)
            y0, x0 = rng.randint(0, gh - sh), rng.randint(0, gw - sw)
            mt = 0 if y0 == 0 else 32
            mb = 0 if (y0 + sh) == gh else 32
            ml = 0 if x0 == 0 else 32
            mr = 0 if (x0 + sw) == gw else 32
            f_sub = bf[:, y0 : y0 + sh, x0 : x0 + sw].unsqueeze(0).to(DEVICE, torch.bfloat16)
            x_sub = bx[:, :, y0 * 2 : (y0 + sh) * 2, x0 * 2 : (x0 + sw) * 2].unsqueeze(0).to(DEVICE, torch.bfloat16)
            t_sub = bt[:, :, y0 * 32 : (y0 + sh) * 32, x0 * 32 : (x0 + sw) * 32].unsqueeze(0).to(DEVICE, torch.float32)
            mar_px = (mt, mb, ml, mr)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = head(f_sub, x_sub, t_offset=0)
        loss, pv = fastframes_video_loss(strip_video_margin(pred, mar_px), strip_video_margin(t_sub, mar_px))
        loss.backward(); torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0); optimizer.step()
        opt_step_count += 1; run_loss += float(loss.detach().item()); run_psnr += pv; run_n += 1

    if r_idx % 25 == 0 or r_idx == 1:
        el = time.time() - t_train
        eta_m = (TOTAL_ROLLOUTS - r_idx) / max(1e-5, (r_idx - start_rollout) / max(1e-3, el)) / 60.0
        print(f"[Rollout {r_idx:4d}/{TOTAL_ROLLOUTS}] loss={run_loss/run_n:.4f} | train_PSNR={run_psnr/run_n:.2f} dB | lr={cur_lr:.2e} | VRAM={torch.cuda.max_memory_allocated()/1e9:.1f}GB | ETA={eta_m:.1f}m")
        run_loss, run_psnr, run_n = 0.0, 0.0, 0

    if r_idx % SAVE_EVERY_ROLLOUTS == 0 or r_idx == TOTAL_ROLLOUTS:
        ev = evaluate_heldout(12)
        history.append({"rollout": r_idx, **ev})
        sec_lbl = "12-Step PDD" if IS_PDD_MODEL else "Base"
        print(f"  [Eval @ {r_idx}] H3 Video Mean PSNR = {ev['mean_psnr']:.2f} dB (8-Step Turbo: {ev['turbo8_psnr']:.2f} dB | {sec_lbl}: {ev['secondary_psnr']:.2f} dB)")
        snap = CKPT_DRIVE / f"minimax_h3_video_pixel_head_r{r_idx:04d}.safetensors"
        save_minimax_h3_pixel_head(head, snap, extra_meta={"base_model": DIT_PATH.name, "teacher_vae": VAE_PATH.name, "rollouts": str(r_idx), "eval_psnr_db": f"{ev['mean_psnr']:.2f}"})
        if ev["mean_psnr"] >= best_eval_psnr or r_idx == TOTAL_ROLLOUTS:
            best_eval_psnr = max(best_eval_psnr, ev["mean_psnr"])
            for dst in (PIXEL_HEADS_DRIVE / "minimax_h3_video_pixel_head.safetensors", VAE_DRIVE / "minimax_h3_video_pixel_head.safetensors", CKPT_DRIVE / "minimax_h3_video_pixel_head_best.safetensors"):
                shutil.copyfile(snap, dst)
            save_bf16_variant(snap, PIXEL_HEADS_DRIVE / "minimax_h3_video_pixel_head_bf16.safetensors")
        torch.save({"arch_version": ARCH_VERSION, "head_width": HEAD_WIDTH, "rollout": r_idx, "opt_step_count": opt_step_count, "head": head.state_dict(), "optimizer": optimizer.state_dict(), "best_eval_psnr": best_eval_psnr, "history": history}, latest_ckpt)

print(f"\nMiniMax-H3 3D Video Pixel Head v2 训练完成。最优视频测试集 PSNR = {best_eval_psnr:.2f} dB")

In [ ]:
# @title 5. 3D 视频跑分评测：Official `MiniMaxH3VideoVAE` (36层 ViT3D, 2.81GB) vs 你的 `MiniMaxH3VideoPixelHead`
import math, time
from pathlib import Path
from IPython.display import Image as IPyImage, Markdown, display
from PIL import Image, ImageDraw
import lpips, numpy as np, torch

lpips_vgg = lpips.LPIPS(net="vgg", verbose=False).to(DEVICE).eval()
EVAL_DRIVE = Path("/content/drive/MyDrive/FastFrames_MiniMaxH3/eval_report")
EVAL_DRIVE.mkdir(parents=True, exist_ok=True)
BEST_HEAD_PATH = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads/minimax_h3_video_pixel_head.safetensors")

eval_head = load_minimax_h3_pixel_head(BEST_HEAD_PATH, device=DEVICE, dtype=torch.bfloat16)
by_domain, per_ph, lat_ph, lat_vae, strips = {}, [], [], [], []

for idx_e, row in enumerate(test_rows[:20]):
    sig = TURBO8_SIGMA_ANCHOR if (idx_e % 2 == 0) else (TURBO12_SIGMA_ANCHOR if IS_PDD_MODEL else 0.300)
    f_g, x_g, _ = engine.rollout(row, sig, 990000 + idx_e)
    x0_hat_vid = x_g.unsqueeze(0).float()

    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        dec = teacher_vae.decode(x0_hat_vid.cpu())[..., :3]
        ref_fhwc = (dec[0] if dec.ndim == 5 else dec).float().clamp(0, 1)
    torch.cuda.synchronize(); lat_vae.append((time.perf_counter() - t0) * 1000.0)

    f_in, x_in = f_g.unsqueeze(0).to(DEVICE, torch.bfloat16), x0_hat_vid.to(DEVICE, torch.bfloat16)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        ph_bcthw = eval_head(f_in, x_in)
    torch.cuda.synchronize(); lat_ph.append((time.perf_counter() - t0) * 1000.0)
    ph_fhwc = ((ph_bcthw[0].float().cpu() + 1.0) * 0.5).clamp(0, 1).permute(1, 2, 3, 0)

    mse_v = max(1e-10, float(((ph_fhwc - ref_fhwc) ** 2).mean().item()))
    psnr_v = -10.0 * math.log10(mse_v)
    # 抽检第 0 帧 (t=0 1f头)、第 11 帧 (4f头)、第 17 帧 (t=5 第二周期 1f头!)
    sample_f = [0, 11, 17]
    with torch.no_grad():
        lp_v = float(lpips_vgg(ph_fhwc[sample_f].movedim(-1, 1).to(DEVICE) * 2 - 1, ref_fhwc[sample_f].movedim(-1, 1).to(DEVICE) * 2 - 1).mean().item())
    by_domain.setdefault(row["domain"], []).append((psnr_v, lp_v))
    per_ph.append((psnr_v, lp_v))
    if len(strips) < 4:
        strips.append((row["domain"], ref_fhwc[sample_f], ph_fhwc[sample_f], psnr_v))

med_vae = sorted(lat_vae)[len(lat_vae) // 2]
med_ph = sorted(lat_ph)[len(lat_ph) // 2]
mean_psnr = sum(x[0] for x in per_ph) / len(per_ph)
mean_lp = sum(x[1] for x in per_ph) / len(per_ph)

lines = [
    "# MiniMax-H3 (Base & 8-Step PDD Turbo) 3D Video Pixel Head — Benchmark Report",
    "",
    "| Video Decoder | File Size | Median Clip Latency (`384×256×22f`) | Speedup vs ViT3D VAE | PSNR (↑ dB) | LPIPS-VGG (↓) |",
    "| :--- | :---: | :---: | :---: | :---: | :---: |",
    f"| **Stock `minimax_h3_video_vae` (36-Layer ViT3D)** | `{VAE_PATH.stat().st_size/(1024*1024):.1f} MiB` | `{med_vae:.1f} ms` | `1.0×` | *(Reference)* | `0.0000` |",
    f"| **Ours: `MiniMaxH3VideoPixelHead`** | **`{BEST_HEAD_PATH.stat().st_size/(1024*1024):.1f} MiB`** | **`{med_ph:.2f} ms`** | **`{med_vae/max(0.05,med_ph):.1f}×`** | **`{mean_psnr:.2f} dB`** | **`{mean_lp:.4f}`** |",
    "",
    "## Per-Motion-Regime Breakdown",
    "",
    "| Motion Regime | Clips | PSNR (↑ dB) | LPIPS-VGG (↓) |",
    "| :--- | :---: | :---: | :---: |",
]
for dom, vals in sorted(by_domain.items()):
    lines.append(f"| `{dom}` | {len(vals)} | `{sum(v[0] for v in vals)/len(vals):.2f} dB` | `{sum(v[1] for v in vals)/len(vals):.4f}` |")

report_md = "\n".join(lines) + "\n"
(EVAL_DRIVE / "BENCHMARK_REPORT.md").write_text(report_md, encoding="utf-8")

fw, fh = strips[0][1].shape[2], strips[0][1].shape[1]
canvas = Image.new("RGB", (3 * fw, len(strips) * 2 * fh), (18, 18, 22))
dr = ImageDraw.Draw(canvas)
for si, (dom, ref3, ph3, pdb) in enumerate(strips):
    for fi in range(3):
        canvas.paste(Image.fromarray((ref3[fi].numpy() * 255.0).round().astype(np.uint8)), (fi * fw, (si * 2) * fh))
        canvas.paste(Image.fromarray((ph3[fi].numpy() * 255.0).round().astype(np.uint8)), (fi * fw, (si * 2 + 1) * fh))
    dr.rectangle([6, (si * 2) * fh + 6, 330, (si * 2) * fh + 28], fill=(0, 0, 0))
    dr.text((12, (si * 2) * fh + 9), f"[{dom}] Stock H3 ViT3D VAE (f=0,11,17)", fill=(255, 255, 255))
    dr.rectangle([6, (si * 2 + 1) * fh + 6, 365, (si * 2 + 1) * fh + 28], fill=(0, 0, 0))
    dr.text((12, (si * 2 + 1) * fh + 9), f"[{dom}] MiniMaxH3PixelHead ({pdb:.1f}dB, {med_ph:.1f}ms)", fill=(120, 255, 160))

strip_path = EVAL_DRIVE / "video_keyframes_comparison.png"
canvas.save(strip_path)
display(Markdown(report_md))
display(IPyImage(filename=str(strip_path)))